# S004 — Ransomware Detection System
## Research-Grade Hybrid EDR Pipeline: Static Memory + Dynamic Telemetry + Orchestrator

**Dataset:** CIC-MalMem2022 (`MalMem2022.csv`) + dynamic VM telemetry (`data/behavioral_raw/`)  
**Layer 1:** Static memory forensics using Random Forest / XGBoost / LightGBM / AE support  
**Layer 2:** Dynamic behavioral 1D-CNN over process telemetry windows  
**Fusion:** Evidence-aware EDR orchestrator with risk states, Layer 1 decay, adaptive CNN thresholds, non-overlapping strikes, and encryption-evidence gating

---

### Improvement integration note

This notebook has been updated to match the improved source code:

- CNN preprocessing preserves per-tick labels when they exist instead of forcing all ransomware-session ticks to `1`.
- CNN training monitors `val_pr_auc`, not `val_recall`, to reduce the almost-always-ransomware failure mode.
- CNN metrics now include PR-AUC, FPR, specificity, and balanced accuracy.
- Layer 2 standalone CNN is treated as a behavioral suspicion signal, not a direct kill authority.
- The EDR orchestrator is now stateful: `SAFE`, `WATCH`, `SUSPICIOUS`, `HIGH_RISK`, `CRITICAL`.
- Hard mitigation requires confidence + persistence + interpretable encryption evidence, usually with Layer 1 support.
- Occlusion and counterfactual helpers are added for explainability and behavioral validation.

**Important:** after these code changes, old notebook outputs should be considered stale until the CNN cache is regenerated and the CNN is retrained.

- v6 adds active-encryption relabeling for all-positive ransomware telemetry sessions and prevents Layer 2 hard-kill while Layer 1 is SAFE/WATCH.


## 0 — Environment Setup

In [ ]:
import os, sys, warnings, importlib
warnings.filterwarnings('ignore')
import matplotlib
matplotlib.use('Agg')          # headless backend — prevents display errors
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
from IPython.display import display, Image
import numpy  as np
import pandas as pd
import seaborn as sns
import optuna
from collections import Counter

# ── Project root ──────────────────────────────────────────────────────────────
NOTEBOOK_DIR = os.path.abspath('')
PROJECT_ROOT = os.path.abspath(os.path.join(NOTEBOOK_DIR, '..'))
SRC_DIR      = os.path.join(PROJECT_ROOT, 'src')
if SRC_DIR not in sys.path:
    sys.path.insert(0, SRC_DIR)
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)
print('Project root:', PROJECT_ROOT)

# ── Reload all modules so edits take effect without restarting kernel ─────────
import src.config
import src.utils
import src.models.data_loader
import src.models.random_forest_model
import src.models.isolation_forest_model
import src.models.autoencoder_model
import src.models.ensemble_model
import src.engine.system_event
import src.evaluate
for mod in [src.config, src.utils, src.models.data_loader,
            src.models.random_forest_model, src.models.isolation_forest_model,
            src.models.autoencoder_model, src.models.ensemble_model,
            src.engine.system_event, src.evaluate]:
    importlib.reload(mod)

from src.config import (
    CICMALMEM_PATHS, PLOTS_DIR, MODELS_DIR, REPORTS_DIR,
    RANDOM_SEED, TEST_SIZE, VAL_SIZE,
    RF_CONFIG, XGB_CONFIG, LGB_CONFIG, IF_CONFIG, AE_CONFIG,
)
from src.utils  import set_seed, Timer, check_class_balance
from src.models.data_loader import (
    load_cicmalmem, get_benign_mask, get_category_series, engineer_features
)
from src.models.random_forest_model    import RansomwareRandomForest
from src.models.isolation_forest_model import RansomwareIsolationForest
from src.models.autoencoder_model      import RansomwareAutoencoder
from src.models.ensemble_model         import RansomwareXGBoost, RansomwareLightGBM, RansomwareEnsemble
from src.engine.system_event           import (
    SystemEvent, make_rule_engine_event,
    SEVERITY_CRITICAL, ACTION_KILL_PROCESS, ACTION_SEND_ALERT
)
from src.evaluate import (
    security_metrics, build_comparison_table,
    plot_combined_curves, plot_threshold_sensitivity,
    run_ablation, build_latex_table, save_results_json
)


# ── Layer 2 — Dynamic CNN imports ─────────────────────────────────────────────
import src.models.cnn_preprocessor
import src.models.cnn_model
import src.engine.edr_orchestrator
for _mod in [src.models.cnn_preprocessor,
             src.models.cnn_model,
             src.engine.edr_orchestrator]:
    importlib.reload(_mod)

from src.models.cnn_preprocessor import (
    preprocess_dynamic_telemetry, discover_raw_csvs, CNN_FEATURE_COLS,
)
from src.models.cnn_model    import RansomwareCNN
from src.engine.edr_orchestrator import EDROrchestrator
from src.engine.system_event import (
    SEVERITY_NONE, SEVERITY_MEDIUM, SEVERITY_HIGH, SEVERITY_CRITICAL,
    ACTION_NO_ACTION, ACTION_LOG_EVENT, ACTION_KILL_PROCESS,
    ACTION_SEND_ALERT,
)

os.makedirs(PLOTS_DIR,   exist_ok=True)
os.makedirs(MODELS_DIR,  exist_ok=True)
os.makedirs(REPORTS_DIR, exist_ok=True)

set_seed(RANDOM_SEED)
plt.style.use('seaborn-v0_8-darkgrid')
plt.rcParams.update({'figure.dpi': 110, 'font.size': 10})
sns.set_palette('husl')

def show(path):
    """Display a saved PNG in the notebook."""
    if os.path.exists(path):
        display(Image(filename=path))
    else:
        print(f'[show] File not found: {path}')

print('\n✓ Environment ready.')
print(f'  NumPy  {np.__version__}  |  Pandas {pd.__version__}')

## 1 — Data Loading & Exploration

In [ ]:
# ── Verify dataset paths ──────────────────────────────────────────────────────
print('Checking dataset paths:')
for p in CICMALMEM_PATHS:
    exists = os.path.exists(p)
    size   = f'{os.path.getsize(p)/1e6:.1f} MB' if exists else 'MISSING'
    print(f'  {"✓" if exists else "✗"}  {os.path.basename(p):<30} {size}')

In [ ]:
# ── Load data with feature engineering ───────────────────────────────────────
print('Loading CIC-MalMem2022 with feature engineering...')
with Timer('Data loading + engineering'):
    X, y = load_cicmalmem(
        file_paths   = CICMALMEM_PATHS,
        verbose      = True,
        add_features = True,
    )
    benign_mask = get_benign_mask(file_paths=CICMALMEM_PATHS)
    categories  = get_category_series(file_paths=CICMALMEM_PATHS)

# Raw features only (for ablation)
raw_cols = [c for c in X.columns if not c.startswith('feat_')]
X_raw    = X[raw_cols].copy()

print(f'\nFinal dataset shape : {X.shape}')
print(f'Raw features        : {len(raw_cols)}')
print(f'Engineered features : {X.shape[1] - len(raw_cols)}')
print(f'Total features      : {X.shape[1]}')
print(f'Benign rows         : {int(benign_mask.sum()):,}  (used for IF + AE training only)')

stats = check_class_balance(y, 'CIC-MalMem2022')

In [ ]:
# Fixed label breakdown — Pandas 3.0 compatible
print('Multi-class breakdown (from Category column):')
print(categories.value_counts().to_string())
print()
# Fix: use np.where instead of .map() for Pandas 3.0 compatibility
label_series = pd.Series(np.where(y == 1, 'Ransomware (1)', 'Non-Ransomware (0)'),
                          index=y.index)
print('Binary labels for training:')
print(label_series.value_counts().to_string())
print(f'\nConfirmed: {int(y.sum()):,} ransomware, {int((y==0).sum()):,} non-ransomware')

In [ ]:
# ── Class distribution plot ───────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

cc = categories.value_counts()
axes[0].bar(cc.index, cc.values,
            color=plt.cm.Set2(np.linspace(0, 1, len(cc))),
            edgecolor='white', linewidth=1.5)
axes[0].set_title('CIC-MalMem2022 — All Categories\n(from Category column)', fontsize=12)
axes[0].set_ylabel('Sample Count')
axes[0].tick_params(axis='x', rotation=20)
for i, v in enumerate(cc.values):
    axes[0].text(i, v + 50, f'{v:,}', ha='center', fontweight='bold', fontsize=9)

bc = Counter(y)
axes[1].bar(['Non-Ransomware (0)', 'Ransomware (1)'],
            [bc[0], bc[1]], color=['steelblue','crimson'],
            edgecolor='white', linewidth=1.5)
axes[1].set_title('Binary Labels Used for Training', fontsize=12)
axes[1].set_ylabel('Sample Count')
for i, v in enumerate([bc[0], bc[1]]):
    axes[1].text(i, v + 50, f'{v:,}', ha='center', fontweight='bold', fontsize=9)

plt.suptitle('Dataset Overview — CIC-MalMem2022', fontsize=13, y=1.02)
plt.tight_layout()
plot_path = os.path.join(PLOTS_DIR, 'class_distribution.png')
fig.savefig(plot_path, dpi=150, bbox_inches='tight')
plt.close(fig)
show(plot_path)

In [ ]:
# Fixed: compute mean differences explicitly
ransomware_mean = X[y == 1].mean()
other_mean      = X[y == 0].mean()
diff = (ransomware_mean - other_mean).abs().sort_values(ascending=False)

print('Top 15 most discriminative features (|mean_ransomware - mean_other|):')
print(diff.head(15).round(4).to_string())
top6 = diff.head(6).index.tolist()

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(16, 8))
axes = axes.flatten()
for i, feat in enumerate(top6):
    ax = axes[i]
    color = '#e74c3c' if feat.startswith('feat_') else '#2980b9'
    ax.hist(X.loc[y==0, feat], bins=50, alpha=0.6, color='steelblue',
            label=f'Non-Ransomware', density=True)
    ax.hist(X.loc[y==1, feat], bins=50, alpha=0.6, color='crimson',
            label=f'Ransomware', density=True)
    prefix = '[ENG] ' if feat.startswith('feat_') else ''
    ax.set_title(f'{prefix}{feat}', fontsize=9,
                 color='#c0392b' if feat.startswith('feat_') else 'black')
    ax.legend(fontsize=7); ax.set_ylabel('Density'); ax.grid(alpha=0.3)

plt.suptitle('Feature Distributions: Non-Ransomware vs Ransomware\n'
             '[ENG] = engineered feature  (red title)', fontsize=12)
plt.tight_layout()
p = os.path.join(PLOTS_DIR, 'feature_distributions.png')
fig.savefig(p, dpi=150, bbox_inches='tight'); plt.close(fig)
show(p)

In [ ]:
# ── Correlation heatmap ───────────────────────────────────────────────────────
top15 = diff.head(15).index.tolist()
corr  = X[top15].corr()
fig, ax = plt.subplots(figsize=(12, 10))
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=True, fmt='.2f', cmap='coolwarm',
            center=0, square=True, linewidths=0.5, vmin=-1, vmax=1, ax=ax)
ax.set_title('Feature Correlation — Top 15 Discriminative Features', fontsize=12)
plt.tight_layout()
p = os.path.join(PLOTS_DIR, 'correlation_heatmap.png')
fig.savefig(p, dpi=150, bbox_inches='tight'); plt.close(fig)
show(p)

## 2 — Random Forest (Recall-Optimised)

In [ ]:
print('='*60)
print('RANDOM FOREST — Supervised | Recall target ≥ 90%')
print('Fix: threshold tuned for recall, NOT F1')
print('Fix: isotonic probability calibration added')
print('='*60)

rf = RansomwareRandomForest(**RF_CONFIG)
rf.feature_names = list(X.columns)

with Timer('RF training'):
    rf_results = rf.train(X, y, test_size=TEST_SIZE, val_size=VAL_SIZE)

print(f'\n✓ RF complete')
print(f'  Recall    : {rf_results["recall"]:.4f}  (target ≥ 0.90)')
print(f'  Precision : {rf_results["precision"]:.4f}')
print(f'  F1        : {rf_results["f1"]:.4f}')
print(f'  ROC-AUC   : {rf_results["roc_auc"]:.4f}')
print(f'  Miss Rate : {rf_results["miss_rate"]:.4f}  (fraction of ransomware missed)')
print(f'  Threshold : {rf.threshold:.4f}  (recall-optimised, NOT default 0.5)')

meets = rf_results['recall'] >= 0.90
print(f'\n  {"✓ MEETS" if meets else "✗ BELOW"} recall target')

In [ ]:
# ── STEP 1: Retrain RF with Optuna-tuned hyperparameters ─────────────────────
print('='*60)
print('RANDOM FOREST — TUNED (Optuna best params)')
print('Best CV recall: 97.88%  |  50 trials')
print('='*60)

# Tuned config — overrides default RF_CONFIG
TUNED_RF_CONFIG = {
    'n_estimators'      : 222,    # tuned (was 500)
    'max_depth'         : 20,     # tuned (was None) ← key change
    'min_samples_split' : 2,      # tuned (was 4)
    'min_samples_leaf'  : 1,      # tuned (was 2)
    'max_features'      : 'sqrt', # same
    'recall_target'     : 0.90,
    'use_smote'         : True,
    'calibrate'         : True,
    'random_state'      : 42,
}

rf_tuned = RansomwareRandomForest(**TUNED_RF_CONFIG)
rf_tuned.feature_names = list(X.columns)

with Timer('RF tuned training'):
    rf_tuned_results = rf_tuned.train(X, y, test_size=TEST_SIZE, val_size=VAL_SIZE)

print(f'\n✓ Tuned RF complete')
print(f'  Recall    : {rf_tuned_results["recall"]:.4f}  (was 0.9229)')
print(f'  Precision : {rf_tuned_results["precision"]:.4f}  (was 0.5436)')
print(f'  F1        : {rf_tuned_results["f1"]:.4f}  (was 0.6842)')
print(f'  ROC-AUC   : {rf_tuned_results["roc_auc"]:.4f}  (was 0.9536)')
print(f'  Miss Rate : {rf_tuned_results["miss_rate"]:.4f}  (was 0.0771)')
print(f'  Threshold : {rf_tuned.threshold:.4f}')

delta_recall = rf_tuned_results["recall"] - rf_results["recall"]
print(f'\n  Delta vs default RF: {delta_recall:+.4f} recall')

rf_tuned.plot_feature_importance(
    top_n=25,
    save_path=os.path.join(PLOTS_DIR, 'rf_tuned_feature_importance.png')
)
plt.close('all')
show(os.path.join(PLOTS_DIR, 'rf_tuned_feature_importance.png'))

rf_tuned.plot_confusion_matrix(
    save_path=os.path.join(PLOTS_DIR, 'rf_tuned_confusion_matrix.png')
)
plt.close('all')
show(os.path.join(PLOTS_DIR, 'rf_tuned_confusion_matrix.png'))

rf_tuned.save(
    model_path  = os.path.join(MODELS_DIR, 'random_forest_tuned.pkl'),
    scaler_path = os.path.join(MODELS_DIR, 'rf_tuned_scaler.pkl'),
)
print('Tuned RF saved.')

In [ ]:
rf.plot_feature_importance(top_n=25, save_path=os.path.join(PLOTS_DIR,'rf_feature_importance.png'))
plt.close('all')
show(os.path.join(PLOTS_DIR,'rf_feature_importance.png'))
print('\nRed bars = engineered features  |  Blue bars = raw Volatility features')

In [ ]:
rf.plot_confusion_matrix(save_path=os.path.join(PLOTS_DIR,'rf_confusion_matrix.png'))
plt.close('all')
show(os.path.join(PLOTS_DIR,'rf_confusion_matrix.png'))

cm = rf_results['confusion_matrix']
TN,FP,FN,TP = cm[0][0],cm[0][1],cm[1][0],cm[1][1]
print(f'\nConfusion matrix interpretation:')
print(f'  True Positives  (ransomware caught)       : {TP:,}')
print(f'  False Negatives (ransomware MISSED) ⚠    : {FN:,}')
print(f'  False Positives (false alarms)            : {FP:,}')
print(f'  True Negatives  (correct non-ransomware)  : {TN:,}')

In [ ]:
rf.plot_roc_curve(save_path=os.path.join(PLOTS_DIR,'rf_roc_curve.png'))
plt.close('all')
show(os.path.join(PLOTS_DIR,'rf_roc_curve.png'))

rf.plot_precision_recall_curve(save_path=os.path.join(PLOTS_DIR,'rf_pr_curve.png'))
plt.close('all')
show(os.path.join(PLOTS_DIR,'rf_pr_curve.png'))

In [ ]:
rf.save(
    model_path  = os.path.join(MODELS_DIR, 'random_forest.pkl'),
    scaler_path = os.path.join(MODELS_DIR, 'rf_scaler.pkl'),
)
print('RF saved.')

## 3 — XGBoost

In [ ]:
print('='*60)
print('XGBOOST — Supervised | Recall target ≥ 90%')
print('Expected: higher recall than RF on tabular data')
print('='*60)

try:
    xgb_model = RansomwareXGBoost(**XGB_CONFIG)
    xgb_model.feature_names = list(X.columns)

    with Timer('XGB training'):
        xgb_results = xgb_model.train(X, y, test_size=TEST_SIZE, val_size=VAL_SIZE)

    print(f'\n✓ XGB complete')
    print(f'  Recall    : {xgb_results["recall"]:.4f}')
    print(f'  Precision : {xgb_results["precision"]:.4f}')
    print(f'  F1        : {xgb_results["f1"]:.4f}')
    print(f'  ROC-AUC   : {xgb_results["roc_auc"]:.4f}')
    print(f'  Threshold : {xgb_model.threshold:.4f}')

    xgb_model.plot_feature_importance(top_n=20,
        save_path=os.path.join(PLOTS_DIR,'xgb_feature_importance.png'))
    plt.close('all')
    show(os.path.join(PLOTS_DIR,'xgb_feature_importance.png'))

    xgb_model.plot_confusion_matrix(
        save_path=os.path.join(PLOTS_DIR,'xgb_confusion_matrix.png'))
    plt.close('all')
    show(os.path.join(PLOTS_DIR,'xgb_confusion_matrix.png'))

    xgb_model.save(
        path        = os.path.join(MODELS_DIR,'xgboost.pkl'),
        scaler_path = os.path.join(MODELS_DIR,'xgb_scaler.pkl'),
    )
    XGB_OK = True

except ImportError:
    print('XGBoost not installed. Run: pip install xgboost')
    XGB_OK = False
    xgb_results = {}

## 4 — LightGBM

In [ ]:
print('='*60)
print('LIGHTGBM — Supervised | Recall target ≥ 90%')
print('Expected: fastest training, competitive recall')
print('='*60)

try:
    lgb_model = RansomwareLightGBM(**LGB_CONFIG)
    lgb_model.feature_names = list(X.columns)

    with Timer('LGB training'):
        lgb_results = lgb_model.train(X, y, test_size=TEST_SIZE, val_size=VAL_SIZE)

    print(f'\n✓ LGB complete')
    print(f'  Recall    : {lgb_results["recall"]:.4f}')
    print(f'  Precision : {lgb_results["precision"]:.4f}')
    print(f'  F1        : {lgb_results["f1"]:.4f}')
    print(f'  ROC-AUC   : {lgb_results["roc_auc"]:.4f}')
    print(f'  Threshold : {lgb_model.threshold:.4f}')

    lgb_model.plot_confusion_matrix(
        save_path=os.path.join(PLOTS_DIR,'lgb_confusion_matrix.png'))
    plt.close('all')
    show(os.path.join(PLOTS_DIR,'lgb_confusion_matrix.png'))

    lgb_model.save(
        path        = os.path.join(MODELS_DIR,'lightgbm.pkl'),
        scaler_path = os.path.join(MODELS_DIR,'lgb_scaler.pkl'),
    )
    LGB_OK = True

except ImportError:
    print('LightGBM not installed. Run: pip install lightgbm')
    LGB_OK = False
    lgb_results = {}

In [ ]:
# ── STEP 2: Retrain XGBoost and LightGBM with Optuna-tuned params ────────────
print('='*60)
print('XGBOOST — TUNED (Optuna, 50 trials)')
print('Best CV recall: 97.38%')
print('='*60)

TUNED_XGB_CONFIG = {
    'n_estimators'    : 590,
    'max_depth'       : 4,           # KEY: shallower than default
    'learning_rate'   : 0.0107,      # slow learner, many trees
    'subsample'       : 0.9985,
    'colsample_bytree': 0.8525,
    'min_child_weight': 4,
    'gamma'           : 0.244,
    'reg_alpha'       : 0.000137,
    'reg_lambda'      : 0.721,
    'recall_target'   : 0.90,
    'use_smote'       : True,
    'calibrate'       : True,
    'random_state'    : 42,
}

xgb_tuned = RansomwareXGBoost(**TUNED_XGB_CONFIG)
xgb_tuned.feature_names = list(X.columns)

with Timer('XGB tuned training'):
    xgb_tuned_results = xgb_tuned.train(X, y, test_size=TEST_SIZE, val_size=VAL_SIZE)

print(f'\nXGB Tuned vs Default:')
print(f'  Recall   : {xgb_tuned_results["recall"]:.4f}  (was 0.8999)')
print(f'  ROC-AUC  : {xgb_tuned_results["roc_auc"]:.4f}  (was 0.9496)')
delta = xgb_tuned_results["recall"] - xgb_results["recall"]
print(f'  Delta    : {delta:+.4f}')

xgb_tuned.save(
    path        = os.path.join(MODELS_DIR, 'xgboost_tuned.pkl'),
    scaler_path = os.path.join(MODELS_DIR, 'xgb_tuned_scaler.pkl'),
)
print('XGB tuned saved.')

print('='*60)
print('LIGHTGBM — TUNED (Optuna, 50 trials)')
print('Best CV recall: 99.75%  ← exceptional')
print('='*60)

TUNED_LGB_CONFIG = {
    'n_estimators'     : 101,        # very few trees needed
    'max_depth'        : 3,          # very shallow — LGB leaf-wise compensates
    'num_leaves'       : 32,
    'learning_rate'    : 0.0210,
    'subsample'        : 0.8898,
    'colsample_bytree' : 0.8294,
    'min_child_samples': 14,
    'reg_alpha'        : 0.000153,
    'reg_lambda'       : 0.00577,
    'recall_target'    : 0.90,
    'use_smote'        : True,
    'calibrate'        : True,
    'random_state'     : 42,
}

lgb_tuned = RansomwareLightGBM(**TUNED_LGB_CONFIG)
lgb_tuned.feature_names = list(X.columns)

with Timer('LGB tuned training'):
    lgb_tuned_results = lgb_tuned.train(X, y, test_size=TEST_SIZE, val_size=VAL_SIZE)

print(f'\nLGB Tuned vs Default:')
print(f'  Recall   : {lgb_tuned_results["recall"]:.4f}  (was 0.9239)')
print(f'  ROC-AUC  : {lgb_tuned_results["roc_auc"]:.4f}  (was 0.9555)')
delta = lgb_tuned_results["recall"] - lgb_results["recall"]
print(f'  Delta    : {delta:+.4f}')

lgb_tuned.save(
    path        = os.path.join(MODELS_DIR, 'lightgbm_tuned.pkl'),
    scaler_path = os.path.join(MODELS_DIR, 'lgb_tuned_scaler.pkl'),
)
print('LGB tuned saved.')

In [ ]:
# ── Split + scale for Optuna tuning ───────────────────────────────────────────
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

RANDOM_STATE = RANDOM_SEED

X_train, X_temp, y_train, y_temp = train_test_split(
    X,
    y,
    test_size=TEST_SIZE + VAL_SIZE,
    stratify=y,
    random_state=RANDOM_STATE
)

test_ratio = TEST_SIZE / (TEST_SIZE + VAL_SIZE)
X_val, X_test, y_val, y_test = train_test_split(
    X_temp,
    y_temp,
    test_size=test_ratio,
    stratify=y_temp,
    random_state=RANDOM_STATE
)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled   = scaler.transform(X_val)
X_test_scaled  = scaler.transform(X_test)

print('Scaled splits:', X_train_scaled.shape, X_val_scaled.shape, X_test_scaled.shape)

In [ ]:
# === Step 1: Re-tune XGBoost properly (100 trials) ===
print('\n=== XGBoost Re-Tuning (100 trials) ===')

def objective_xgb_v2(trial):
    params = {
        'n_estimators': trial.suggest_int('n_estimators', 150, 700),
        'max_depth': trial.suggest_int('max_depth', 3, 10),
        'learning_rate': trial.suggest_float('learning_rate', 1e-3, 0.2, log=True),
        'subsample': trial.suggest_float('subsample', 0.6, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.6, 1.0),
        'gamma': trial.suggest_float('gamma', 0.0, 5.0),
        'min_child_weight': trial.suggest_int('min_child_weight', 1, 12),
        'reg_alpha': trial.suggest_float('reg_alpha', 1e-8, 10.0, log=True),
        'reg_lambda': trial.suggest_float('reg_lambda', 1e-6, 30.0, log=True)
    }

    # Safe fixed threshold for tuning
    mdl = RansomwareXGBoost(random_state=RANDOM_STATE, **params)
    mdl.threshold = 0.5
    try:
        mdl.train(X_train, y_train)
        X_test_sc = mdl.scaler.transform(X_test)
        p = mdl.predict_proba_batch(X_test_sc)
        if p is None or len(p) == 0:
            return 0.0
        y_pred = (p >= 0.5).astype(int)
        m = security_metrics(y_test, y_pred, p, 'XGB v2')
        # prioritize recall heavily, but keep precision/F1 pressure
        score = 0.75 * m['recall'] + 0.25 * m['f1']
        return float(score)
    except Exception:
        return 0.0

study_xgb_v2 = optuna.create_study(direction='maximize')
study_xgb_v2.optimize(objective_xgb_v2, n_trials=100, show_progress_bar=False)

best_xgb_v2 = study_xgb_v2.best_params
print('Best XGB params v2:', best_xgb_v2)

# Refit best model + threshold tuning
xgb_tuned_v2 = RansomwareXGBoost(**best_xgb_v2, random_state=RANDOM_STATE)
xgb_tuned_v2.train(X_train, y_train)
X_val_sc = xgb_tuned_v2.scaler.transform(X_val)
p_val = xgb_tuned_v2.predict_proba_batch(X_val_sc)
xgb_tuned_v2.threshold = xgb_tuned_v2._tune_recall(p_val, y_val, verbose=False)

X_test_sc = xgb_tuned_v2.scaler.transform(X_test)
p_test = xgb_tuned_v2.predict_proba_batch(X_test_sc)
y_pred = (p_test >= xgb_tuned_v2.threshold).astype(int)

xgb_tuned_v2_results = security_metrics(y_test, y_pred, p_test, 'XGB v2')
xgb_tuned_v2_results.update({'y_test': y_test, 'test_proba': p_test})

print(f"XGB v2 threshold: {xgb_tuned_v2.threshold:.4f}")
print(f"XGB v2 recall   : {xgb_tuned_v2_results['recall']:.4f}")
print(f"XGB v2 precision: {xgb_tuned_v2_results['precision']:.4f}")
print(f"XGB v2 f1       : {xgb_tuned_v2_results['f1']:.4f}")

In [ ]:
# === Step 2: Build final best ensemble from strongest tuned models ===
print('\n=== Building Final Best Ensemble ===')

# choose best base models
rf_best_for_ens  = rf_tuned
xgb_best_for_ens = xgb_tuned_v2
lgb_best_for_ens = lgb_tuned

# build ensemble using the project class
ens_best_v2 = RansomwareEnsemble(recall_target=0.90)
ens_best_v2.add_model('random_forest', rf_best_for_ens)
ens_best_v2.add_model('xgboost',       xgb_best_for_ens)
ens_best_v2.add_model('lightgbm',      lgb_best_for_ens)

ens_best_v2_results = ens_best_v2.evaluate(X, y, test_size=TEST_SIZE)
print(f"Final Ensemble v2 threshold: {ens_best_v2.threshold:.4f}")
print(f"Final Ensemble v2 recall   : {ens_best_v2_results['recall']:.4f}")
print(f"Final Ensemble v2 precision: {ens_best_v2_results['precision']:.4f}")
print(f"Final Ensemble v2 f1       : {ens_best_v2_results['f1']:.4f}")

In [ ]:
# === Step 3: Final definitive comparison table (v3) ===
print('\n=== Final Comparison Table v3 ===')

all_metrics_final_v3 = []
models_probas_final_v3 = {}
operating_pts_final_v3 = {}

def register_final_v3(name, res, model_obj):
    if not res:
        return
    y_t = np.array(res.get('y_test', []))

    # IF handling
    if 'test_scores' in res and 'test_proba' not in res:
        raw_scores = np.array(res['test_scores'])
        p = np.clip(-raw_scores, 0, 1)
        pred = (raw_scores < model_obj.threshold).astype(int)
        t_used = float(model_obj.threshold)
    else:
        p = np.array(res.get('test_proba', []))
        t_used = float(getattr(model_obj, 'threshold', 0.5))
        pred = (p >= t_used).astype(int)

    if len(y_t) == 0 or len(p) == 0:
        return

    m = security_metrics(y_t, pred, p, name)
    all_metrics_final_v3.append(m)
    models_probas_final_v3[name] = (y_t, p)
    operating_pts_final_v3[name] = t_used

def maybe_register(name, res_var, model_var):
    if res_var not in globals() or model_var not in globals():
        print(f"  Skipping {name} — missing {res_var} or {model_var}")
        return
    register_final_v3(name, globals()[res_var], globals()[model_var])

maybe_register('RF (default)', 'rf_results', 'rf')
maybe_register('RF (tuned)', 'rf_tuned_results', 'rf_tuned')
maybe_register('XGBoost (default)', 'xgb_results', 'xgb_model')
maybe_register('XGBoost (tuned)', 'xgb_tuned_v2_results', 'xgb_tuned_v2')
maybe_register('LightGBM (default)', 'lgb_results', 'lgb_model')
maybe_register('LightGBM (tuned)', 'lgb_tuned_results', 'lgb_tuned')
maybe_register('Ensemble (default)', 'ens_results', 'ens')
maybe_register('Ensemble (best)', 'ens_best_v2_results', 'ens_best_v2')
maybe_register('Isolation Forest', 'if_results', 'iso')
if globals().get('AE_OK') and 'ae_results' in globals() and 'ae' in globals():
    register_final_v3('Autoencoder', ae_results, ae)

comp_final_v3 = build_comparison_table(all_metrics_final_v3)

highlight_cols = [c for c in ['recall', 'f1', 'roc_auc', 'pr_auc'] if c in comp_final_v3.columns]
if highlight_cols:
    display(comp_final_v3.style.highlight_max(axis=0, subset=highlight_cols))
else:
    display(comp_final_v3)

In [ ]:
# === Step 4: Final combined ROC plot including all corrected models ===
print('\n=== Plotting Final ROC Curves (v3) ===')

import matplotlib.pyplot as plt
from sklearn.metrics import roc_curve, roc_auc_score

plt.figure(figsize=(10, 8))
for name, (y_t, p) in models_probas_final_v3.items():
    fpr, tpr, _ = roc_curve(y_t, p)
    auc_val = roc_auc_score(y_t, p)
    plt.plot(fpr, tpr, label=f'{name} (AUC={auc_val:.3f})')

plt.plot([0, 1], [0, 1], '--', linewidth=1)
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('ROC Curves - Final Corrected Model Set')
plt.legend(loc='lower right', fontsize=8)
plt.grid(alpha=0.25)
plt.tight_layout()
plot_path = os.path.join(PLOTS_DIR, 'roc_curves_final_v3.png')
plt.savefig(plot_path, dpi=300)
plt.close()
show(plot_path)

## 5 — Isolation Forest (Benign-Only Training — Fixed)

In [ ]:
print('='*60)
print('ISOLATION FOREST — Unsupervised Anomaly Detector')
print('FIX v1: trained on Benign-only rows (not mixed non-ransomware)')
print('FIX v2: contamination=0.165 (actual ransomware ratio, was 0.05)')
print('Role: Layer 2 — catches zero-day ransomware not in training labels')
print('='*60)
print(f'\nBenign rows available for training: {int(benign_mask.sum()):,}')
print(f'Total rows                         : {len(y):,}')

iso = RansomwareIsolationForest(**IF_CONFIG)
iso.feature_names = list(X.columns)

with Timer('IF training'):
    if_results = iso.train(X, y, benign_mask=benign_mask, test_size=TEST_SIZE)

print(f'\n✓ IF complete')
print(f'  Recall    : {if_results["recall"]:.4f}')
print(f'  Precision : {if_results["precision"]:.4f}')
print(f'  F1        : {if_results["f1"]:.4f}')
print(f'  ROC-AUC   : {if_results["roc_auc"]:.4f}')
print(f'  Threshold : {iso.threshold:.4f}')
print()
print('Note: IF ROC-AUC of ~0.70–0.80 on static memory data is expected and correct.')
print('IF value is zero-day detection — it was NOT designed to match RF accuracy.')

In [ ]:
iso.plot_score_distribution(
    save_path=os.path.join(PLOTS_DIR,'if_score_distribution.png'))
plt.close('all')
show(os.path.join(PLOTS_DIR,'if_score_distribution.png'))

iso.plot_confusion_matrix(
    save_path=os.path.join(PLOTS_DIR,'if_confusion_matrix.png'))
plt.close('all')
show(os.path.join(PLOTS_DIR,'if_confusion_matrix.png'))

iso.save(
    model_path  = os.path.join(MODELS_DIR,'isolation_forest.pkl'),
    scaler_path = os.path.join(MODELS_DIR,'if_scaler.pkl'),
)
print('IF saved.')

## 6 — Deep Autoencoder (Replaces LSTM)

In [ ]:
print('='*60)
print('DEEP AUTOENCODER — Unsupervised Anomaly Detector')
print('Replaces the LSTM which was wrong for static tabular data.')
print('Architecture: 70 → 128 → 64 → 32 → 16 → 32 → 64 → 128 → 70')
print('Trained on Benign-only rows.')
print('Detects ransomware via high reconstruction error.')
print('='*60)

AE_OK = False
ae_results = {}

try:
    import tensorflow as tf
    print(f'TensorFlow {tf.__version__} found.')

    ae = RansomwareAutoencoder(**AE_CONFIG)
    ae.feature_names = list(X.columns)
    ae.n_features    = len(X.columns)

    with Timer('AE training'):
        ae_results = ae.train(X, y, benign_mask=benign_mask, test_size=TEST_SIZE)

    print(f'\n✓ AE complete')
    print(f'  Recall    : {ae_results["recall"]:.4f}')
    print(f'  Precision : {ae_results["precision"]:.4f}')
    print(f'  F1        : {ae_results["f1"]:.4f}')
    print(f'  ROC-AUC   : {ae_results["roc_auc"]:.4f}')
    print(f'  Threshold : {ae.threshold:.6f}')

    ae.plot_training_history(
        save_path=os.path.join(PLOTS_DIR,'ae_training_history.png'))
    plt.close('all')
    show(os.path.join(PLOTS_DIR,'ae_training_history.png'))

    ae.plot_error_distribution(
        save_path=os.path.join(PLOTS_DIR,'ae_error_distribution.png'))
    plt.close('all')
    show(os.path.join(PLOTS_DIR,'ae_error_distribution.png'))

    ae.plot_confusion_matrix(
        save_path=os.path.join(PLOTS_DIR,'ae_confusion_matrix.png'))
    plt.close('all')
    show(os.path.join(PLOTS_DIR,'ae_confusion_matrix.png'))

    ae.save(
        model_dir   = os.path.join(MODELS_DIR,'autoencoder'),
        scaler_path = os.path.join(MODELS_DIR,'ae_scaler.pkl'),
    )
    AE_OK = True

except ImportError:
    print('TensorFlow not installed. pip install tensorflow')

In [ ]:
# ── STEP 3: Best ensemble using all tuned models ──────────────────────────────
print('='*60)
print('BEST ENSEMBLE V7 — Recall-oriented tuned RF + tuned LGB + tuned XGB')
print('='*60)

ens_best = RansomwareEnsemble(
    weights={
        'random_forest': 0.45,   # strong supervised model
        'lightgbm'     : 0.40,   # strong supervised model
        'xgboost'      : 0.15,   # supporting supervised model
        'autoencoder'  : 0.00,   # novelty booster, excluded from recall score
    },
    recall_target=0.90,
    any_model_threshold=0.85,
    min_supervised_votes=2,
)

# Add tuned versions where available
ens_best.add_model('random_forest', rf_tuned)
ens_best.add_model('xgboost',       xgb_tuned)
ens_best.add_model('lightgbm',      lgb_tuned)
if AE_OK:
    ens_best.add_model('autoencoder', ae)

with Timer('Best ensemble evaluation'):
    ens_best_results = ens_best.evaluate(X, y, test_size=TEST_SIZE)

print(f'\nBest Ensemble V7 results:')
print(f'  Recall   : {ens_best_results["recall"]:.4f}')
print(f'  Precision: {ens_best_results["precision"]:.4f}')
print(f'  F1       : {ens_best_results["f1"]:.4f}')
print(f'  ROC-AUC  : {ens_best_results["roc_auc"]:.4f}')
print(f'  Fusion   : {ens_best_results.get("fusion_rule", "weighted")}')


In [ ]:
# ── Fix: store AE results in format compatible with comparison table ──────────
# Root cause of AE zeros: ae.threshold is in reconstruction-error scale (~1.98)
# but test_proba is normalised to [0,1]. Applying 1.98 to [0,1] → all False → recall=0.
# Fix: also compute and store 'ae_normalized_threshold' in [0,1] space.

if AE_OK:
    from sklearn.metrics import (
        roc_auc_score, average_precision_score,
        recall_score, precision_score, f1_score, precision_recall_curve,
    )

    ae_errors = np.array(ae_results.get('test_errors', []))
    ae_y_test = np.array(ae_results.get('y_test', []))

    if len(ae_errors) > 0 and len(np.unique(ae_y_test)) > 1:
        # 1. Normalise errors → [0,1] probabilities
        max_err  = np.percentile(ae_errors, 99)
        ae_proba = np.clip(ae_errors / max(max_err, 1e-9), 0, 1)
        ae_results['test_proba'] = ae_proba

        # 2. ROC-AUC + avg precision on normalised probabilities
        ae_roc = roc_auc_score(ae_y_test, ae_proba)
        ae_avg = average_precision_score(ae_y_test, ae_proba)
        ae_results['roc_auc']       = float(ae_roc)
        ae_results['avg_precision'] = float(ae_avg)

        # 3. Find optimal threshold in [0,1] space (max F1 on precision-recall curve)
        precs, recs, thrs = precision_recall_curve(ae_y_test, ae_proba)
        precs_t, recs_t   = precs[:-1], recs[:-1]
        f1s = 2 * precs_t * recs_t / (precs_t + recs_t + 1e-9)
        best_idx = int(np.argmax(f1s))
        ae_norm_threshold = float(thrs[best_idx])
        ae_results['ae_normalized_threshold'] = ae_norm_threshold

        # 4. Compute corrected metrics at normalised threshold
        ae_pred = (ae_proba >= ae_norm_threshold).astype(int)
        ae_rec  = float(recall_score(ae_y_test,    ae_pred, zero_division=0))
        ae_prec = float(precision_score(ae_y_test, ae_pred, zero_division=0))
        ae_f1   = float(f1_score(ae_y_test,        ae_pred, zero_division=0))

        # 5. Store corrected scalar metrics so register() reads the right values
        ae_results['recall']    = ae_rec
        ae_results['precision'] = ae_prec
        ae_results['f1']        = ae_f1
        ae_results['miss_rate'] = round(1 - ae_rec, 4)

        print(f'[AE] ROC-AUC            : {ae_roc:.4f}')
        print(f'[AE] Avg Precision       : {ae_avg:.4f}')
        print(f'[AE] Normalised threshold: {ae_norm_threshold:.4f}  (was: {ae.threshold:.4f} in error scale)')
        print(f'[AE] Recall @norm thr    : {ae_rec:.4f}')
        print(f'[AE] Precision @norm thr : {ae_prec:.4f}')
        print(f'[AE] F1 @norm thr        : {ae_f1:.4f}')
    else:
        print('[AE] No test errors or single class — skipping AE correction.')


In [ ]:
# ── AE metrics cross-check (using raw reconstruction-error threshold) ────────
# This cell is for reference only — shows recall at ae.threshold.
# The comparison tables in cells 38/40 use the normalised [0,1] threshold
# stored in ae_results['ae_normalized_threshold'] (computed in cell 34).

if AE_OK and 'test_errors' in ae_results:
    from sklearn.metrics import recall_score, precision_score, f1_score
    ae_errors_raw = np.array(ae_results['test_errors'])
    ae_y_raw      = np.array(ae_results['y_test'])

    ae_pred_raw = (ae_errors_raw > ae.threshold).astype(int)
    rec_raw     = recall_score(ae_y_raw,    ae_pred_raw, zero_division=0)
    prec_raw    = precision_score(ae_y_raw, ae_pred_raw, zero_division=0)
    f1_raw      = f1_score(ae_y_raw,        ae_pred_raw, zero_division=0)

    print(f'[AE reference] Raw reconstruction-error threshold: {ae.threshold:.6f}')
    print(f'  Recall    : {rec_raw:.4f}')
    print(f'  Precision : {prec_raw:.4f}')
    print(f'  F1        : {f1_raw:.4f}')
    print()
    print(f'[AE in tables] Normalised threshold: {ae_results.get("ae_normalized_threshold", "N/A"):.4f}')
    print(f'  Recall    : {ae_results.get("recall", 0):.4f}')
    print(f'  Precision : {ae_results.get("precision", 0):.4f}')
    print(f'  F1        : {ae_results.get("f1", 0):.4f}')
    print(f'  ROC-AUC   : {ae_results.get("roc_auc", 0):.4f}')


## 7 — Weighted Ensemble

In [ ]:
print('='*60)
print('RECALL-ORIENTED ENSEMBLE V7 — weighted score + vote/high-score safety rule')
print('Weights: RF=0.45, LGB=0.40, XGB=0.15, AE=0.00')
print('='*60)

ens = RansomwareEnsemble(recall_target=0.90, any_model_threshold=0.85, min_supervised_votes=2)
ens.add_model('random_forest', rf)
if XGB_OK: ens.add_model('xgboost',   xgb_model)
if LGB_OK: ens.add_model('lightgbm',  lgb_model)
if AE_OK:  ens.add_model('autoencoder', ae)

if len(ens.models) >= 2:
    with Timer('Ensemble evaluation'):
        ens_results = ens.evaluate(X, y, test_size=TEST_SIZE)

    print(f'\n✓ Ensemble complete')
    print(f'  Recall    : {ens_results["recall"]:.4f}')
    print(f'  Precision : {ens_results["precision"]:.4f}')
    print(f'  F1        : {ens_results["f1"]:.4f}')
    print(f'  ROC-AUC   : {ens_results["roc_auc"]:.4f}')
    print(f'  Threshold : {ens.threshold:.4f}')
    ENS_OK = True
else:
    print('Need ≥2 trained models for ensemble.')
    ens_results = {}; ENS_OK = False


In [ ]:
# ── Build comparison table for all trained models ─────────────────────────────
from sklearn.metrics import roc_curve

all_metrics   = []
models_probas = {}
operating_pts = {}

def register(name, res, model_obj):
    if not res:
        print(f'  Skipping {name} — no results')
        return
    y_t = np.array(res.get('y_test', []))

    # IF: scores stored as test_scores (anomaly scores), not test_proba
    if 'test_scores' in res and 'test_proba' not in res:
        raw_sc = np.array(res['test_scores'])
        p      = np.clip(-raw_sc, 0, 1)   # invert: more negative = more anomalous
        t_raw  = float(model_obj.threshold)
        pred   = (raw_sc < t_raw).astype(int)
        t      = t_raw
    else:
        p = np.array(res.get('test_proba', []))
        # AE FIX: ae.threshold is in reconstruction-error scale, not [0,1].
        # Use ae_normalized_threshold (stored by cell 34) for the [0,1] probabilities.
        if 'ae_normalized_threshold' in res:
            t = float(res['ae_normalized_threshold'])
        else:
            t = float(getattr(model_obj, 'threshold', 0.5))
        pred = (p >= t).astype(int)

    if len(y_t) == 0 or len(p) == 0:
        print(f'  Skipping {name} — empty arrays')
        return
    m = security_metrics(y_t, pred, p, name)
    all_metrics.append(m)
    models_probas[name] = (y_t, p)
    operating_pts[name] = t
    print(f'  Registered: {name}  recall={m["recall"]:.4f}')

print('Registering models...')
register('Random Forest (default)', rf_results,       rf)
register('Random Forest (tuned)',   rf_tuned_results, rf_tuned)
if xgb_results:
    register('XGBoost',             xgb_results,      xgb_model)
if lgb_results:
    register('LightGBM',            lgb_results,      lgb_model)
register('Isolation Forest',        if_results,       iso)
if AE_OK and ae_results:
    register('Autoencoder',         ae_results,       ae)
if 'ens_results' in dir() and ens_results:
    register('Ensemble',            ens_results,      ens)

print(f'\nTotal models registered: {len(all_metrics)}')
comp_df = build_comparison_table(all_metrics)


## 8 — Model Comparison

In [ ]:
# ── FINAL comparison table with all tuned models ─────────────────────────────
all_metrics_final   = []
models_probas_final = {}
operating_pts_final = {}

def register_final(name, res, model_obj):
    if not res:
        print(f'  Skipping {name} — no results')
        return
    y_t = np.array(res.get('y_test', []))

    # IF: uses raw anomaly scores
    if 'test_scores' in res and 'test_proba' not in res:
        raw_sc = np.array(res['test_scores'])
        p      = np.clip(-raw_sc, 0, 1)
        t_raw  = float(model_obj.threshold)
        pred   = (raw_sc < t_raw).astype(int)
        t      = t_raw
    else:
        p = np.array(res.get('test_proba', []))
        # AE FIX: use normalised threshold from cell 34
        if 'ae_normalized_threshold' in res:
            t = float(res['ae_normalized_threshold'])
        else:
            t = float(getattr(model_obj, 'threshold', 0.5))

        # IF fallback (when test_proba was manually set)
        if name == 'Isolation Forest' and 'test_scores' in res:
            raw_sc = np.array(res['test_scores'])
            pred   = (raw_sc < model_obj.threshold).astype(int)
        else:
            pred = (p >= t).astype(int)

    if len(y_t) == 0 or len(p) == 0:
        print(f'  Skipping {name} — empty arrays')
        return
    m = security_metrics(y_t, pred, p, name)
    all_metrics_final.append(m)
    models_probas_final[name] = (y_t, p)
    operating_pts_final[name] = t
    print(f'  Registered: {name}  recall={m["recall"]:.4f}')

register_final('RF (default)',         rf_results,         rf)
register_final('RF (tuned)',           rf_tuned_results,   rf_tuned)
register_final('XGBoost (default)',    xgb_results,        xgb_model)
if 'xgb_tuned_results' in dir() and xgb_tuned_results:
    register_final('XGBoost (tuned)',  xgb_tuned_results,  xgb_tuned)
register_final('LightGBM (default)',  lgb_results,        lgb_model)
if 'lgb_tuned_results' in dir() and lgb_tuned_results:
    register_final('LightGBM (tuned)',lgb_tuned_results,  lgb_tuned)
if 'ens_results' in dir() and ens_results:
    register_final('Ensemble (default)', ens_results,     ens)
if 'ens_best_results' in dir() and ens_best_results:
    register_final('Ensemble (best)',  ens_best_results,   ens_best)
register_final('Isolation Forest',    if_results,         iso)
if AE_OK and ae_results:
    register_final('Autoencoder',     ae_results,         ae)

comp_final = build_comparison_table(all_metrics_final)

latex_final = build_latex_table(
    all_metrics_final,
    save_path=os.path.join(REPORTS_DIR, 'model_comparison_final.tex')
)
print('\nFinal LaTeX table:')
print(latex_final)


In [ ]:
# ── Bar chart comparison ──────────────────────────────────────────────────────
metrics_to_plot = ['recall','precision','f1','roc_auc']
model_names     = [m['model'] for m in all_metrics]
colors          = plt.cm.Set1(np.linspace(0, 0.8, len(model_names)))
x               = np.arange(len(metrics_to_plot))
w               = 0.8 / max(len(model_names), 1)

fig, ax = plt.subplots(figsize=(14, 6))
for i, (m_dict, color) in enumerate(zip(all_metrics, colors)):
    vals = [m_dict[k] for k in metrics_to_plot]
    bars = ax.bar(x + i*w, vals, w, label=m_dict['model'], color=color,
                  alpha=0.85, edgecolor='white')
    for bar, v in zip(bars, vals):
        ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.008,
                f'{v:.3f}', ha='center', va='bottom', fontsize=7.5, fontweight='bold')

ax.axhline(0.90, color='red', linestyle='--', alpha=0.4, lw=1.5, label='90% recall target')
ax.set_xticks(x + w*(len(model_names)-1)/2)
ax.set_xticklabels(['Recall','Precision','F1','ROC-AUC'], fontsize=12)
ax.set_ylabel('Score', fontsize=12)
ax.set_ylim(0, 1.15)
ax.set_title('Model Performance Comparison — CIC-MalMem2022', fontsize=13)
ax.legend(fontsize=9, loc='upper right'); ax.grid(axis='y', alpha=0.3)

p = os.path.join(PLOTS_DIR,'model_comparison.png')
fig.savefig(p, dpi=150, bbox_inches='tight'); plt.close(fig)
show(p)

In [ ]:
# ── Combined ROC + PR curves ──────────────────────────────────────────────────
plot_combined_curves(
    models_probas,
    operating_pts = operating_pts,
    save_path     = os.path.join(PLOTS_DIR,'combined_roc_pr.png')
)
plt.close('all')
show(os.path.join(PLOTS_DIR,'combined_roc_pr.png'))

In [ ]:
# ── Threshold sensitivity ─────────────────────────────────────────────────────
supervised_probas = {k: v for k, v in models_probas.items()
                     if k not in ('Isolation Forest', 'Autoencoder')}
if supervised_probas:
    plot_threshold_sensitivity(
        supervised_probas,
        save_path=os.path.join(PLOTS_DIR,'threshold_sensitivity.png')
    )
    plt.close('all')
    show(os.path.join(PLOTS_DIR,'threshold_sensitivity.png'))

## 9 — Ablation Study: Raw vs Engineered Features

In [ ]:
print('='*60)
print('ABLATION STUDY')
print('Random Forest trained with RAW features (55) vs ENGINEERED (70)')
print('Quantifies the contribution of the 15 ratio/interaction features')
print('='*60)

ablation_df = run_ablation(
    X_raw     = X_raw,
    X_eng     = X,
    y         = y,
    save_path = os.path.join(REPORTS_DIR, 'ablation_results.csv'),
    verbose   = True
)
print(ablation_df[['model','recall','precision','f1','roc_auc']].to_string(index=False))

In [ ]:
# ── Ablation bar chart ────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 4, figsize=(16, 5))
metrics_ab = ['recall','precision','f1','roc_auc']
labels_ab  = ['Raw (55)', 'Engineered (70)']
colors_ab  = ['#95a5a6','#2980b9']
for ax, met in zip(axes, metrics_ab):
    vals = ablation_df[met].astype(float).tolist()
    bars = ax.bar(labels_ab, vals, color=colors_ab, edgecolor='white', linewidth=1.5)
    for bar, v in zip(bars, vals):
        ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.003,
                f'{v:.4f}', ha='center', va='bottom', fontweight='bold', fontsize=10)
    ax.set_title(met.upper(), fontsize=11)
    ax.set_ylim(max(0, min(vals)-0.05), min(1.05, max(vals)+0.05))
    ax.grid(axis='y', alpha=0.3)
    ax.tick_params(axis='x', rotation=10)

plt.suptitle('Ablation Study: Impact of Engineered Features on Random Forest',
             fontsize=13)
plt.tight_layout()
p = os.path.join(PLOTS_DIR,'ablation_bar.png')
fig.savefig(p, dpi=150, bbox_inches='tight'); plt.close(fig)
show(p)

## 10 — SystemEvent Translation Demo

In [ ]:
print('='*60)
print('SYSTEMEVENT TRANSLATION DEMO')
print('Each model outputs a SystemEvent — standard interface for Mitigation')
print('='*60)

# Build a test feature snapshot with elevated ransomware signals
test_feat = {col: float(X[col].median()) for col in X.columns}
# Inject ransomware-like values
ransomware_signals = {
    'malfind.ninjections'   : 8.0,
    'malfind.commitCharge'  : 6.0,
    'ldrmodules.not_in_load': 5.0,
    'psxview.not_in_pslist' : 4.0,
    'callbacks.ncallbacks'  : 3.0,
    'feat_injection_density': 2.5,
    'feat_hidden_dll_ratio' : 1.8,
    'feat_inject_x_hidden'  : 40.0,
}
for k, v in ransomware_signals.items():
    if k in test_feat:
        test_feat[k] = v

TEST_PID  = 4821
TEST_NAME = 'suspicious_proc.exe'
print(f'\nSimulated process: PID={TEST_PID}, name={TEST_NAME}')
print('Injected elevated ransomware signals into feature snapshot')

In [ ]:
import json

# RF → SystemEvent
e_rf = rf.predict(test_feat, pid=TEST_PID, process_name=TEST_NAME)
print('RANDOM FOREST EVENT:')
print(f'  {e_rf}')
print(f'  Alert      = {e_rf.alert}')
print(f'  Severity   = {e_rf.severity}')
print(f'  Confidence = {e_rf.confidence:.2%}')
print(f'  Actions    = {e_rf.recommended_actions}')
print()

In [ ]:
# IF → SystemEvent
e_if = iso.predict(test_feat, pid=TEST_PID, process_name=TEST_NAME)
print('ISOLATION FOREST EVENT:')
print(f'  {e_if}')
print(f'  Alert        = {e_if.alert}')
print(f'  Anomaly Score= {e_if.anomaly_score:.4f}  (more negative = more suspicious)')
print(f'  Actions      = {e_if.recommended_actions}')
print()

In [ ]:
# Rule Engine → SystemEvent (simulating RULE-001)
e_rule = make_rule_engine_event(
    pid=TEST_PID, process_name='cmd.exe',
    rule_id='RULE-001', severity=SEVERITY_CRITICAL,
    response_actions=[ACTION_KILL_PROCESS, ACTION_SEND_ALERT],
    features={'command': 'vssadmin delete shadows /all /quiet'},
    description='Shadow copy deletion detected — RULE-001'
)
print('RULE ENGINE EVENT (RULE-001: vssadmin delete shadows):')
print(f'  {e_rule}')
print(f'  Confidence = {e_rule.confidence:.0%}  (always 100% for CRITICAL rules)')
print()
print('JSON serialisation (for logging):')
print(json.dumps(e_rule.to_dict(), indent=2))

## 11 — Layered Detection Demo

In [ ]:
print('='*60)
print('LAYERED DETECTION ARCHITECTURE')
print('Layer 1: Rule Engine  — instant, ~100% precision, known commands')
print('Layer 2: Isolation Forest — anomaly screening, catches zero-days')
print('Layer 3: RF + XGB + LGB — high-confidence classification')
print('='*60)

DANGEROUS_COMMANDS = [
    'vssadmin delete shadows',
    'bcdedit /set recoveryenabled no',
    'wbadmin delete catalog',
    'wmic shadowcopy delete',
]

def layered_detect(features, pid=0, process_name='unknown', cmdline=''):
    # Layer 1: Rules — instant, 100% precision
    for cmd in DANGEROUS_COMMANDS:
        if cmd.lower() in cmdline.lower():
            event = make_rule_engine_event(
                pid=pid, process_name=process_name,
                rule_id='RULE-001', severity='CRITICAL',
                response_actions=[ACTION_KILL_PROCESS, ACTION_SEND_ALERT],
                features=features,
                description=f'Dangerous command: {cmd}'
            )
            print(f'  [L1 RULE  ] FIRED → {event}')
            return event

    # Layer 2: Isolation Forest — anomaly screening
    e_if = iso.predict(features, pid, process_name)
    if not e_if.alert:
        print(f'  [L2 IF    ] Normal. Score={e_if.anomaly_score:.4f} — no alert')
        from src.engine.system_event import SystemEvent, SEVERITY_NONE, ACTION_NO_ACTION
        return SystemEvent(alert=False, severity=SEVERITY_NONE,
                           model_source='Layered',
                           recommended_actions=[ACTION_NO_ACTION])

    print(f'  [L2 IF    ] Anomaly detected. Score={e_if.anomaly_score:.4f}')

    # Layer 3: RF confirmation — only alert if RF agrees at >= 75%
    e_rf = rf_tuned.predict(features, pid, process_name)
    print(f'  [L3 RF    ] Confidence={e_rf.confidence:.2%}')

    if e_rf.confidence >= 0.75:
        print(f'  [L3 HIGH  ] IF + RF both agree → HIGH confidence ransomware alert')
        return e_rf
    elif e_rf.confidence >= 0.40:
        # IF flagged, RF partially agrees — log and monitor, do NOT suspend
        print(f'  [L3 MEDIUM] Partial agreement — logging and monitoring only')
        from src.engine.system_event import SystemEvent, SEVERITY_MEDIUM, ACTION_LOG_EVENT
        return SystemEvent(
            alert=False,          # do NOT fire a process action
            severity=SEVERITY_MEDIUM,
            model_source='Layered',
            confidence=e_rf.confidence,
            pid=pid, process_name=process_name,
            recommended_actions=[ACTION_LOG_EVENT],
            description=f'IF anomaly not confirmed by RF (conf={e_rf.confidence:.2%}). Monitoring.'
        )
    else:
        # RF says low probability — IF false positive, ignore
        print(f'  [L3 IGNORE] RF={e_rf.confidence:.2%} < 40% — IF false positive, no alert')
        from src.engine.system_event import SystemEvent, SEVERITY_NONE, ACTION_NO_ACTION
        return SystemEvent(alert=False, severity=SEVERITY_NONE,
                           model_source='Layered',
                           recommended_actions=[ACTION_NO_ACTION])

# Test 1: Benign process
print('\n--- TEST 1: Normal notepad.exe ---')
benign_feat = {col: float(X[col].quantile(0.25)) for col in X.columns}
res1 = layered_detect(benign_feat, pid=1001, process_name='notepad.exe')
print(f'  Final: Alert={res1.alert}\n')

# Test 2: Dangerous command
print('--- TEST 2: Process running vssadmin delete shadows ---')
res2 = layered_detect(benign_feat, pid=4821, process_name='cmd.exe',
                       cmdline='vssadmin delete shadows /all /quiet')
print(f'  Final: Alert={res2.alert}, Severity={res2.severity}\n')

# Test 3: Suspicious feature pattern
print('--- TEST 3: Suspicious process (ransomware-like features) ---')
res3 = layered_detect(test_feat, pid=TEST_PID, process_name=TEST_NAME)
print(f'  Final: Alert={res3.alert}, Confidence={res3.confidence:.2%}')

## 12 — Save Full Results & LaTeX Table

In [ ]:
# Save full results — run this only after Cell 29 above has completed
if all_metrics:
    save_results_json(
        all_metrics,
        save_path  = os.path.join(REPORTS_DIR, 'evaluation_results.json'),
        extra_meta = {
            'dataset'       : 'CIC-MalMem2022',
            'n_samples'     : int(len(y)),
            'n_features'    : int(X.shape[1]),
            'recall_target' : 0.90,
            'hpo_best_recall': 0.9788,
        }
    )
    latex = build_latex_table(
        all_metrics,
        save_path=os.path.join(REPORTS_DIR, 'model_comparison.tex')
    )
    print('\nLaTeX table (paste into your report):')
    print(latex)
else:
    print('ERROR: all_metrics is empty. Run the comparison table cell (Cell 29 fix) first.')

## 16 — Cross-Family Generalisation Test

In [ ]:
# ═══════════════════════════════════════════════════════════════════════
# SECTION 16 — CROSS-FAMILY GENERALISATION TEST
#
# Why this matters:
#   Training and testing on the same ransomware families is easy.
#   Real-world deployment means encountering NEW families.
#   This test measures how well the model generalises.
#
# Protocol:
#   For each ransomware family in the dataset:
#     1. Train RF on ALL OTHER ransomware families + all benign/spyware/trojan
#     2. Test on the held-out family ONLY
#     3. Report recall on that family
#   This is "leave-one-family-out" cross-validation.
# ═══════════════════════════════════════════════════════════════════════

from sklearn.ensemble        import RandomForestClassifier
from sklearn.preprocessing   import StandardScaler
from sklearn.metrics         import recall_score, precision_score, f1_score
from imblearn.over_sampling  import SMOTE
import warnings
warnings.filterwarnings('ignore')

print('='*65)
print('CROSS-FAMILY GENERALISATION TEST')
print('Leave-One-Family-Out Protocol')
print('Answers: Can the model detect ransomware it has NEVER seen?')
print('='*65)

# ── Get per-family category labels aligned with X and y ──────────────────────
# categories has same length as the dataset — align with X index
cat_array = categories.values[:len(y)]
y_array   = y.values

# Get unique ransomware families
ransomware_families = [c for c in np.unique(cat_array)
                       if 'ransomware' in c.lower()]
print(f'\nRansomware families in dataset: {ransomware_families}')
print()

family_results = []

for held_out_family in ransomware_families:
    # ── Split: held-out family vs everything else ─────────────────────────────
    held_out_mask = (cat_array == held_out_family)
    train_mask    = ~held_out_mask

    X_train_cf = X.values[train_mask]
    y_train_cf = y_array[train_mask]
    X_test_cf  = X.values[held_out_mask]
    y_test_cf  = y_array[held_out_mask]

    n_test_ransomware = int(y_test_cf.sum())
    n_train_ransomware = int(y_train_cf.sum())

    print(f'  Held-out: {held_out_family}')
    print(f'    Test samples (all ransomware)   : {len(X_test_cf)}')
    print(f'    Train ransomware (other families): {n_train_ransomware}')

    if n_train_ransomware == 0:
        print(f'    SKIP — no other ransomware in training set')
        continue

    # ── Scale ─────────────────────────────────────────────────────────────────
    sc = StandardScaler()
    X_train_sc = sc.fit_transform(X_train_cf)
    X_test_sc  = sc.transform(X_test_cf)

    # ── SMOTE ─────────────────────────────────────────────────────────────────
    k = min(5, n_train_ransomware - 1)
    try:
        X_train_sm, y_train_sm = SMOTE(
            random_state=42, k_neighbors=k
        ).fit_resample(X_train_sc, y_train_cf)
    except Exception:
        X_train_sm, y_train_sm = X_train_sc, y_train_cf

    # ── Train RF with tuned params ─────────────────────────────────────────────
    clf = RandomForestClassifier(
        n_estimators     = 222,    # tuned params
        max_depth        = 20,
        min_samples_split= 2,
        min_samples_leaf = 1,
        max_features     = 'sqrt',
        class_weight     = 'balanced',
        random_state     = 42,
        n_jobs           = -1,
    )
    clf.fit(X_train_sm, y_train_sm)

    # ── Evaluate at recall-optimised threshold 0.16 ───────────────────────────
    proba = clf.predict_proba(X_test_sc)[:, 1]
    pred  = (proba >= 0.16).astype(int)  # use same threshold as main model

    recall    = recall_score(y_test_cf, pred, zero_division=0)
    precision = precision_score(y_test_cf, pred, zero_division=0)
    f1        = f1_score(y_test_cf, pred, zero_division=0)

    # Since all test samples ARE ransomware, recall = fraction correctly classified
    family_results.append({
        'family'           : held_out_family,
        'n_test'           : len(X_test_cf),
        'n_train_ransomware': n_train_ransomware,
        'recall'           : recall,
        'precision'        : precision,
        'f1'               : f1,
        'meets_target'     : recall >= 0.88,
    })

    status = '✓' if recall >= 0.88 else '✗'
    print(f'    Recall: {recall:.4f}  Precision: {precision:.4f}  F1: {f1:.4f}  {status}')
    print()

# ── Summary table ──────────────────────────────────────────────────────────────
if family_results:
    cf_df = pd.DataFrame(family_results)
    print('\n' + '='*65)
    print('CROSS-FAMILY GENERALISATION RESULTS SUMMARY')
    print('='*65)
    print(cf_df[['family','n_test','recall','precision','f1','meets_target']].to_string(index=False))
    print()
    print(f'  Mean recall across families : {cf_df["recall"].mean():.4f}')
    print(f'  Min recall (hardest family) : {cf_df["recall"].min():.4f}  '
          f'({cf_df.loc[cf_df["recall"].idxmin(), "family"]})')
    print(f'  Max recall (easiest family) : {cf_df["recall"].max():.4f}  '
          f'({cf_df.loc[cf_df["recall"].idxmax(), "family"]})')
    families_passing = int(cf_df["meets_target"].sum())
    print(f'  Families meeting ≥88% target: {families_passing}/{len(cf_df)}')
    print()
    if cf_df["recall"].mean() >= 0.85:
        print('  ✓ GOOD generalisation — model detects unseen ransomware families')
    else:
        print('  ✗ Poor generalisation — model overfits to seen families')

    # Save results
    cf_df.to_csv(os.path.join(REPORTS_DIR, 'cross_family_results.csv'), index=False)
    print(f'\n  Results saved → reports/cross_family_results.csv')

In [ ]:
# ── Cross-family bar chart ─────────────────────────────────────────────────────
if family_results:
    cf_df = pd.DataFrame(family_results)

    fig, ax = plt.subplots(figsize=(10, 5))
    colors = ['#27ae60' if r >= 0.88 else '#e74c3c' for r in cf_df['recall']]
    bars   = ax.bar(cf_df['family'], cf_df['recall'], color=colors,
                    edgecolor='white', linewidth=1.5)

    for bar, val in zip(bars, cf_df['recall']):
        ax.text(bar.get_x() + bar.get_width()/2,
                bar.get_height() + 0.008,
                f'{val:.3f}', ha='center', va='bottom',
                fontweight='bold', fontsize=11)

    ax.axhline(0.88, color='red', linestyle='--', lw=2,
               label='88% target threshold')
    ax.set_xlabel('Held-Out Ransomware Family', fontsize=12)
    ax.set_ylabel('Recall (fraction of unseen family detected)', fontsize=12)
    ax.set_title('Cross-Family Generalisation Test\n'
                 'Green = meets target  |  Red = below target\n'
                 '(Trained WITHOUT this family, tested ON this family only)',
                 fontsize=12)
    ax.set_ylim(0, 1.12)
    ax.legend(fontsize=10)
    ax.grid(axis='y', alpha=0.3)
    ax.tick_params(axis='x', rotation=15)

    p = os.path.join(PLOTS_DIR, 'cross_family_generalisation.png')
    fig.savefig(p, dpi=150, bbox_inches='tight')
    plt.close(fig)
    show(p)

In [ ]:
# ── Per-family analysis with standard RF (for comparison) ─────────────────────
print('='*65)
print('PER-FAMILY DETECTION RATE (Standard RF — in-distribution)')
print('Trained on ALL families, tested on each family subset')
print('='*65)

# Use the already-trained tuned RF model
per_family_results = []
X_test_rf = rf_tuned_results['X_test']
y_test_rf  = np.array(rf_tuned_results['y_test'])
proba_rf   = rf_tuned_results['test_proba']

# Get categories aligned with test set indices
# We need to reconstruct which test samples belong to which family
# Use the same split seed to reproduce the test set indices
from sklearn.model_selection import train_test_split

_, X_test_orig, _, y_test_idx = train_test_split(
    X, y, test_size=TEST_SIZE, stratify=y, random_state=42)

test_categories = categories.values[:len(y)][y_test_idx.index]

for fam in np.unique(test_categories):
    fam_mask = (test_categories == fam)
    if fam_mask.sum() == 0:
        continue

    y_fam    = y_test_rf[fam_mask]
    p_fam    = proba_rf[fam_mask]
    n_ransom = int(y_fam.sum())
    n_total  = int(fam_mask.sum())

    if n_ransom == 0:
        category_type = 'Benign/Spyware/Trojan'
        # FPR for non-ransomware families
        pred = (p_fam >= rf_tuned.threshold).astype(int)
        fpr  = float(pred.mean())
        per_family_results.append({
            'family': fam, 'type': category_type,
            'n_samples': n_total, 'recall': None, 'fpr': fpr
        })
    else:
        category_type = 'Ransomware'
        pred   = (p_fam >= rf_tuned.threshold).astype(int)
        recall = recall_score(y_fam, pred, zero_division=0)
        per_family_results.append({
            'family': fam, 'type': category_type,
            'n_samples': n_total, 'recall': recall, 'fpr': None
        })

pf_df = pd.DataFrame(per_family_results)
print('\nRansomware families — Recall:')
ransom_df = pf_df[pf_df['type']=='Ransomware'].sort_values('recall')
print(ransom_df[['family','n_samples','recall']].to_string(index=False))

print('\nNon-Ransomware families — False Positive Rate:')
nonransom_df = pf_df[pf_df['type']!='Ransomware'].sort_values('fpr', ascending=False)
print(nonransom_df[['family','n_samples','fpr']].to_string(index=False))

pf_df.to_csv(os.path.join(REPORTS_DIR, 'per_family_results.csv'), index=False)
print(f'\nSaved → reports/per_family_results.csv')

In [ ]:
# ── Final project summary table — generated from actual result objects ─────────
print('='*75)
print('FINAL PROJECT RESULTS SUMMARY — S004 Ransomware Detection System')
print('Dataset: CIC-MalMem2022  |  values generated from current notebook run')
print('='*75)

summary_rows = []

def add_summary_row(name, model_type, result_obj, model_obj=None, training_time='run-dependent'):
    if not result_obj:
        return
    row = {
        'Model': name,
        'Type': model_type,
        'Recall': result_obj.get('recall', np.nan),
        'Precision': result_obj.get('precision', np.nan),
        'F1': result_obj.get('f1', np.nan),
        'ROC-AUC': result_obj.get('roc_auc', np.nan),
        'Threshold': getattr(model_obj, 'threshold', result_obj.get('threshold', np.nan)) if model_obj is not None else result_obj.get('threshold', np.nan),
        'Training Time': training_time,
    }
    try:
        row['Meets ≥90% Recall'] = '✓' if float(row['Recall']) >= 0.90 else '✗'
    except Exception:
        row['Meets ≥90% Recall'] = 'N/A'
    summary_rows.append(row)

add_summary_row('RF (default)', 'Supervised', globals().get('rf_results', {}), globals().get('rf', None))
add_summary_row('RF (tuned)', 'Supervised', globals().get('rf_tuned_results', {}), globals().get('rf_tuned', None))
add_summary_row('XGBoost (default)', 'Supervised', globals().get('xgb_results', {}), globals().get('xgb_model', None))
add_summary_row('XGBoost (tuned)', 'Supervised', globals().get('xgb_tuned_results', {}), globals().get('xgb_tuned', None))
add_summary_row('LightGBM (default)', 'Supervised', globals().get('lgb_results', {}), globals().get('lgb_model', None))
add_summary_row('LightGBM (tuned)', 'Supervised', globals().get('lgb_tuned_results', {}), globals().get('lgb_tuned', None))
add_summary_row('Ensemble (default/V7)', 'Supervised ensemble', globals().get('ens_results', {}), globals().get('ens', None))
add_summary_row('Ensemble (best/V7)', 'Supervised ensemble', globals().get('ens_best_results', {}), globals().get('ens_best', None))
add_summary_row('Isolation Forest', 'Unsupervised', globals().get('if_results', {}), globals().get('iso', None))
add_summary_row('Autoencoder (AE)', 'Unsupervised', globals().get('ae_results', {}), globals().get('ae', None))

summary_df = pd.DataFrame(summary_rows)
pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', 24)
pd.set_option('display.width', 140)
print(summary_df.to_string(index=False))

print('\nKey interpretation:')
print('  • This table is generated from actual metrics in the current run; no hard-coded ensemble recall.')
print('  • Individual supervised Layer 1 models should remain above the recall target.')
print('  • V7 ensemble uses recall-oriented fusion, so its recall should no longer collapse below the base models.')
print('  • Layer 2 CNN is a behavioral suspicion model; process-level EDR actions are evaluated in Section 17.')




## 17 — Improved Layer 2 and Hybrid EDR Orchestrator

This section replaces the older CNN/orchestrator experiment. It matches the updated source code and addresses the main failures observed in the previous run:

1. **CNN label pollution:** per-tick labels are preserved instead of forcing all ticks in ransomware folders to `1`.
2. **Recall-only optimization:** CNN training now monitors `val_pr_auc`, reducing the risk of an almost-always-positive model.
3. **CNN-only hard blocking:** Layer 2 is now a behavioral suspicion signal; the orchestrator decides mitigation.
4. **Overlapping strike issue:** strike counting can be non-overlapping so windows `(1--8)`, `(2--9)`, and `(3--10)` are not treated as independent proof.
5. **Missing behavioral proof:** hard mitigation now requires encryption-like evidence such as read/write volume, write/read ratio, CPU/write coupling, and write intensity.


### 17.1 — Paths, Configuration, and Improvement Flags

In [ ]:
# ── 17.1 Paths & improved configuration ─────────────────────────────────────
BEHAVIORAL_RAW_DIR = os.path.join(PROJECT_ROOT, 'data', 'behavioral_raw')
NPZ_CACHE          = os.path.join(PROJECT_ROOT, 'data', 'behavioral_sequences.npz')
CNN_SAVE_DIR       = os.path.join(MODELS_DIR, 'cnn')
CNN_SCALER_PATH    = os.path.join(MODELS_DIR, 'cnn_scaler.pkl')

CNN_WINDOW_SIZE = 8
CNN_STEP_SIZE   = 1

# IMPORTANT: force reprocessing after the V7 label/window-policy fix. Old caches preserve the
# old all-positive ransomware-session labels.
FORCE_REPROCESS_DYNAMIC = True

# A window is positive only if enough ticks inside it are active-encryption ticks.
CNN_MAJORITY_FRACTION = 0.25

# V7: if ransomware CSV labels are all-positive, derive active-encryption tick
# labels from read/write/CPU/open-file evidence instead of training on folder identity.
CNN_RELABEL_POLICY = 'auto_evidence'  # options: preserve, auto_evidence, evidence
LABEL_EVIDENCE_THRESHOLDS = {
    'write_min_bytes': 64 * 1024,
    'read_min_bytes':  16 * 1024,
    'ratio_min': 0.25,
    'cpu_write_min': 1_000_000.0,
    'write_intensity_min': 128.0,
    'cpu_min': 8.0,
    'open_files_min': 2.0,
    'min_hits': 2,
    'dilate_before_ticks': 1,
    'dilate_after_ticks': 2,
    'smooth_ticks': 0,
}

# CNN is a suspicion model; mitigation is handled by the orchestrator.
CNN_RECALL_TARGET   = 0.85
CNN_THRESHOLD_FLOOR = 0.05

# Orchestrator policy: evidence-aware, stateful, and safer than CNN-only kill.
CNN_N_CONSECUTIVE       = 3
CNN_SUSPEND_THRESHOLD   = 0.85
CNN_KILL_THRESHOLD      = 0.97
NON_OVERLAPPING_STRIKES = True
REQUIRE_ENCRYPTION_EVIDENCE = True

# V7 risk accumulator: restores detection when the CNN is bursty but
# interpretable encryption evidence persists.
L2_RISK_DECAY = 0.85
L2_RISK_SOFT_THRESHOLD = 0.50
L2_EVIDENCE_PERSISTENCE_WINDOWS = 5
L2_STRONG_EVIDENCE_HITS = 5

# Conservative runtime evidence thresholds. These are stricter than the label
# thresholds because mitigation needs stronger proof than training relabeling.
EVIDENCE_THRESHOLDS = {
    'write_min_bytes': 256 * 1024,
    'read_min_bytes':  64 * 1024,
    'ratio_min': 0.50,
    'cpu_write_min': 5_000_000.0,
    'write_intensity_min': 512.0,
    'cpu_min': 15.0,
    'open_files_min': 5.0,
    'min_hits': 3,
}

from src.models.cnn_preprocessor import CNN_FEATURE_COLS_DERIVED
CNN_FEATURE_COLS = CNN_FEATURE_COLS_DERIVED

print('Improved Layer 2 / EDR configuration:')
print(f'  window_size              : {CNN_WINDOW_SIZE} ticks')
print(f'  step_size                : {CNN_STEP_SIZE}')
print(f'  force_reprocess_dynamic  : {FORCE_REPROCESS_DYNAMIC}')
print(f'  relabel_policy           : {CNN_RELABEL_POLICY}')
print(f'  majority_pos_fraction    : {CNN_MAJORITY_FRACTION}')
print(f'  CNN recall target        : {CNN_RECALL_TARGET}')
print(f'  CNN threshold floor      : {CNN_THRESHOLD_FLOOR}')
print(f'  non-overlapping strikes  : {NON_OVERLAPPING_STRIKES}')
print(f'  L2 risk accumulator      : decay={L2_RISK_DECAY}, soft_thr={L2_RISK_SOFT_THRESHOLD}')
print(f'  require evidence gate    : {REQUIRE_ENCRYPTION_EVIDENCE}')
print(f'  suspend / kill thresholds: {CNN_SUSPEND_THRESHOLD} / {CNN_KILL_THRESHOLD}')
print(f'  n_features               : {len(CNN_FEATURE_COLS)}')

print('\nChecking behavioral_raw subfolders:')
import glob as _glob
DYNAMIC_OK = True
for sub in ['cerber', 'ryuk', 'wannacry', 'data']:
    folder = os.path.join(BEHAVIORAL_RAW_DIR, sub)
    if os.path.isdir(folder):
        n_csv  = len(_glob.glob(os.path.join(folder, '*.csv')))
        label  = 'ransomware' if sub != 'data' else 'benign'
        print(f'  ✓  {sub:<12} ({label}) — {n_csv} CSVs')
    else:
        print(f'  ✗  {sub:<12} MISSING')
        DYNAMIC_OK = False

NPZ_EXISTS = os.path.exists(NPZ_CACHE)
print(f'\n  .npz cache : {"exists" if NPZ_EXISTS else "not found"}')



### 17.2 — Dynamic Preprocessing With Preserved Per-Tick Labels

The updated `cnn_preprocessor.py` now does the following:

- preserves the CSV `label` column when it exists;
- uses folder labels only as a fallback;
- prints a label audit showing benign/ransomware tick distribution and mixed sessions;
- rejects older caches using cache versioning;
- uses `RobustScaler` fit on the training ticks only;
- uses `GroupKFold` by session/process group to avoid leakage.


In [ ]:
# ── 17.2 Preprocessing with V7 active-encryption + 25% window label policy ────────────────
CNN_DATA_OK = False
CNN_TRAINED = False
cnn_data    = {}

if DYNAMIC_OK or NPZ_EXISTS:
    print('='*72)
    print('LAYER 2 PREPROCESSING — V7 relabeling + temporal dilation')
    print('  policy  : preserve mixed CSV labels; relabel all-positive ransomware sessions')
    print('            into active-encryption ticks using read/write/CPU evidence + local dilation')
    print('  scaler  : RobustScaler fit on train ticks only')
    print('  split   : GroupKFold by session/process group')
    print('  cache   : versioned; force_reprocess recommended after label changes')
    print('='*72)

    import importlib
    import src.models.cnn_preprocessor as _prep
    importlib.reload(_prep)
    from src.models.cnn_preprocessor import preprocess_dynamic_telemetry, discover_raw_csvs

    with Timer('CNN preprocessing'):
        cnn_data = preprocess_dynamic_telemetry(
            behavioral_raw_dir    = BEHAVIORAL_RAW_DIR,
            window_size           = CNN_WINDOW_SIZE,
            step_size             = CNN_STEP_SIZE,
            n_folds               = 5,
            scaler_path           = CNN_SCALER_PATH,
            npz_cache             = NPZ_CACHE,
            feature_cols          = CNN_FEATURE_COLS,
            majority_pos_fraction = CNN_MAJORITY_FRACTION,
            relabel_policy        = CNN_RELABEL_POLICY,
            label_evidence_thresholds = LABEL_EVIDENCE_THRESHOLDS,
            force_reprocess       = FORCE_REPROCESS_DYNAMIC,
            verbose               = True,
        )

    n_ben_tr = int((cnn_data['y_train'] == 0).sum())
    n_ran_tr = int((cnn_data['y_train'] == 1).sum())
    n_ben_va = int((cnn_data['y_val']   == 0).sum())
    n_ran_va = int((cnn_data['y_val']   == 1).sum())
    n_ben_te = int((cnn_data['y_test']  == 0).sum())
    n_ran_te = int((cnn_data['y_test']  == 1).sum())

    print('\nData quality summary:')
    print(f'  Train windows : benign={n_ben_tr:,}  ransomware={n_ran_tr:,}')
    print(f'  Val windows   : benign={n_ben_va:,}  ransomware={n_ran_va:,}')
    print(f'  Test windows  : benign={n_ben_te:,}  ransomware={n_ran_te:,}')
    print(f'  input_shape   : {cnn_data["input_shape"]}')
    print(f'  class_weight  : {cnn_data["class_weight"]}')

    if n_ran_tr == 0 or n_ran_va == 0 or n_ran_te == 0:
        print('\n✗ CRITICAL: at least one split has zero ransomware windows. Check label thresholds.')
        CNN_DATA_OK = False
    else:
        print('\n✓ Dynamic dataset is usable for CNN training.')
        print('  Important: ransomware windows should no longer be 100% of ransomware sessions, but should not be too sparse.')
        CNN_DATA_OK = True
else:
    print('⚠  No behavioral data or usable cache found. CNN section will be skipped.')



### 17.3 — CNN Training With `val_pr_auc` Monitoring

The improved `RansomwareCNN` monitors `val_pr_auc` instead of `val_recall`. This matters because recall-only early stopping can select a model that predicts ransomware too often, which was one of the likely causes of the high false-positive rate.

The threshold is still tuned on validation data, but it is not a mitigation decision. It is only the CNN suspicion threshold. The orchestrator decides whether the evidence justifies watch, suspend, or kill.


In [ ]:
# ── 17.3 Train improved CNN ─────────────────────────────────────────────────
cnn = None
cnn_results = {}

if CNN_DATA_OK:
    try:
        import tensorflow as tf
        import importlib
        import src.models.cnn_model as _cnn_mod
        importlib.reload(_cnn_mod)
        from src.models.cnn_model import RansomwareCNN

        print('='*72)
        print('LAYER 2 — 1D-CNN behavioral monitor')
        print(f'TensorFlow: {tf.__version__}')
        print('Training monitor: val_pr_auc')
        print('Layer 2 role: behavioral suspicion, not direct hard-kill authority')
        print('='*72)

        CNN_CONFIG = {
            'window_size'     : CNN_WINDOW_SIZE,
            'n_features'      : cnn_data['input_shape'][1],
            'filters_1'       : 64,
            'filters_2'       : 128,
            'dropout_rate'    : 0.4,
            'learning_rate'   : 5e-4,
            'l2_reg'          : 1e-4,
            'recall_target'   : CNN_RECALL_TARGET,
            'threshold_floor' : CNN_THRESHOLD_FLOOR,
            'epochs'          : 80,
            'batch_size'      : 64,
            'random_state'    : RANDOM_SEED,
        }
        print('CNN_CONFIG =')
        for k, v in CNN_CONFIG.items():
            print(f'  {k:<16}: {v}')

        cnn = RansomwareCNN(**CNN_CONFIG)
        os.makedirs(CNN_SAVE_DIR, exist_ok=True)

        with Timer('CNN training'):
            cnn_results = cnn.train(
                data      = cnn_data,
                model_dir = CNN_SAVE_DIR,
            )

        # Attach inference-time preprocessing artifacts for orchestrator use.
        cnn.scaler       = cnn_data['scaler']
        cnn.feature_cols = cnn_data['feature_cols']
        cnn.save(model_dir=CNN_SAVE_DIR, scaler_path=CNN_SCALER_PATH)

        cm = cnn_results.get('confusion_matrix', [[0, 0], [0, 0]])
        tn, fp = cm[0][0], cm[0][1]
        fn, tp = cm[1][0], cm[1][1]

        print('\n✓ CNN training complete')
        print(f'  threshold          : {cnn.threshold:.6f}')
        print(f'  recall             : {cnn_results["recall"]:.4f}')
        print(f'  precision          : {cnn_results["precision"]:.4f}')
        print(f'  F1                 : {cnn_results["f1"]:.4f}')
        print(f'  ROC-AUC            : {cnn_results["roc_auc"]:.4f}')
        print(f'  PR-AUC             : {cnn_results["pr_auc"]:.4f}')
        print(f'  FPR                : {cnn_results["fpr"]:.4f}')
        print(f'  specificity        : {cnn_results["specificity"]:.4f}')
        print(f'  balanced_accuracy  : {cnn_results["balanced_accuracy"]:.4f}')
        print(f'  confusion matrix   : TN={tn} FP={fp} FN={fn} TP={tp}')

        if cnn_results['fpr'] > 0.20:
            print('\n⚠ FPR remains high. Treat CNN as suspicion only and tune with hard-benign controls.')
        CNN_TRAINED = True

    except ImportError:
        print('⚠ TensorFlow is not installed. CNN training skipped.')
        CNN_TRAINED = False
    except Exception as e:
        import traceback; traceback.print_exc()
        CNN_TRAINED = False
else:
    print('Skipping CNN training — run preprocessing first and check label audit.')


### 17.4 — CNN Curves, Confusion Matrix, and Metric Registration

In [ ]:
# ── 17.4 Visualise and register CNN metrics ─────────────────────────────────
if CNN_TRAINED:
    hist_path = os.path.join(PLOTS_DIR, 'cnn_training_history_improved.png')
    cnn.plot_training_history(save_path=hist_path)
    plt.close('all')
    show(hist_path)

    cm_path = os.path.join(PLOTS_DIR, 'cnn_confusion_matrix_improved.png')
    cnn.plot_confusion_matrix(save_path=cm_path)
    plt.close('all')
    show(cm_path)

    from sklearn.metrics import classification_report
    y_pred_cnn = (cnn_results['test_proba'] >= cnn.threshold).astype(int)
    print('\nClassification Report — Improved CNN Layer 2:')
    print(classification_report(
        cnn_results['y_test'], y_pred_cnn,
        target_names=['Benign', 'Ransomware'],
        digits=4,
        zero_division=0,
    ))

    try:
        m_cnn = security_metrics(
            cnn_results['y_test'], y_pred_cnn, cnn_results['test_proba'],
            'CNN Layer2 Improved (behavioral)'
        )
        # Add metrics returned by the improved CNN that security_metrics may not include.
        for extra_key in ['pr_auc', 'fpr', 'specificity', 'balanced_accuracy']:
            m_cnn[extra_key] = cnn_results.get(extra_key)

        if 'all_metrics' in globals():
            all_metrics = [m for m in all_metrics if m.get('model') != 'CNN Layer2 Improved (behavioral)']
            all_metrics.append(m_cnn)
            print('✓ Improved CNN registered in all_metrics')
    except Exception as e:
        print(f'Could not register CNN in all_metrics: {e}')
else:
    print('CNN not trained — visualisation skipped.')


### 17.5 — Hybrid EDR Orchestrator v5

The improved orchestrator uses:

- Layer 1 memory probability as a **forensic prior**;
- risk states: `SAFE`, `WATCH`, `SUSPICIOUS`, `HIGH_RISK`, `CRITICAL`;
- Layer 1 decay so old memory evidence weakens over time;
- adaptive CNN thresholds based on Layer 1 suspicion;
- non-overlapping strikes;
- encryption-evidence gating before mitigation;
- soft block before hard block unless Layer 1 is critical or CNN evidence is extremely strong.


In [ ]:
# ── 17.5 Build improved EDR orchestrator ────────────────────────────────────
import importlib
import src.engine.edr_orchestrator as _edr_mod
importlib.reload(_edr_mod)
EDROrchestrator = _edr_mod.EDROrchestrator
RiskState       = _edr_mod.RiskState

EDR_OK = False
orchestrator = None

if CNN_TRAINED:
    # Prefer the tuned RF from earlier notebook cells. Fall back to standard RF.
    static_model_for_edr = globals().get('rf_tuned', None) or globals().get('rf', None)

    if static_model_for_edr is None:
        print('⚠ No trained Layer 1 static model found in memory. Run RF sections first.')
    else:
        # Ensure feature schema is available for the orchestrator feature_dict path.
        static_model_for_edr.feature_names = list(X.columns)
        print('Layer 1 model selected for EDR:', type(static_model_for_edr).__name__)
        print(f'  feature count : {len(static_model_for_edr.feature_names)}')
        print(f'  threshold     : {getattr(static_model_for_edr, "threshold", 0.5):.4f}')

        try:
            orchestrator = EDROrchestrator(
                static_model               = static_model_for_edr,
                cnn_model                  = cnn,
                l1_threshold               = None,  # use trained static model threshold when available
                n_consecutive              = CNN_N_CONSECUTIVE,
                suspend_threshold          = CNN_SUSPEND_THRESHOLD,
                kill_threshold             = CNN_KILL_THRESHOLD,
                l1_watch_threshold         = 0.10,
                l1_suspicious_threshold    = 0.35,
                l1_high_threshold          = 0.75,
                l1_critical_threshold      = 0.90,
                min_cnn_threshold          = 0.20,
                adaptive_alpha             = 0.35,
                require_encryption_evidence= REQUIRE_ENCRYPTION_EVIDENCE,
                non_overlapping_strikes    = NON_OVERLAPPING_STRIKES,
                l1_decay_tau_seconds       = 60.0,
                l1_decay_floor             = 0.10,
                evidence_thresholds        = EVIDENCE_THRESHOLDS,
                risk_decay                 = L2_RISK_DECAY,
                risk_soft_threshold        = L2_RISK_SOFT_THRESHOLD,
                evidence_persistence_windows = L2_EVIDENCE_PERSISTENCE_WINDOWS,
                strong_evidence_hits       = L2_STRONG_EVIDENCE_HITS,
                verbose                    = True,
            )
            EDR_OK = True
            print('\n✓ EDR Orchestrator v7 ready')
            print(f'  L1 decision threshold       : {orchestrator.l1_threshold:.4f}')
            print(f'  base CNN threshold          : {cnn.threshold:.6f}')
            print(f'  non-overlapping strikes     : {orchestrator.non_overlapping_strikes}')
            print(f'  encryption evidence required: {orchestrator.require_encryption_evidence}')
            print(f'  suspend / kill thresholds   : {orchestrator.suspend_threshold} / {orchestrator.kill_threshold}')
            print(f'  risk accumulator            : decay={orchestrator.risk_decay}, soft_thr={orchestrator.risk_soft_threshold}')
        except Exception as e:
            import traceback; traceback.print_exc()
else:
    print('CNN not trained — run cells 17.2 and 17.3 first.')



### 17.6 — Orchestrator Regression Tests

These tests replace the older direct CNN-block tests. The expected behavior is now safer:

- benign Notepad-like activity should be `SAFE` or at most `WATCH`, not hard-killed;
- behavioral ransomware should be stopped by Layer 2 only after evidence and persistence;
- memory-forensics ransomware should be escalated by Layer 1, with engineered features recomputed from raw values rather than manually injected inconsistently.


In [ ]:
# ── 17.6 Helper functions for orchestrator tests ─────────────────────────────
from src.models.data_loader import RAW_FEATURE_COLUMNS, engineer_features

def print_event(event, title='SystemEvent'):
    print('\n' + '='*72)
    print(title)
    print('='*72)
    print(f'alert       : {event.alert}')
    print(f'severity    : {event.severity}')
    print(f'source      : {event.model_source}')
    print(f'confidence  : {event.confidence}')
    print(f'actions     : {event.recommended_actions}')
    print(f'description : {event.description}')
    if getattr(event, 'features', None):
        print('features summary:')
        for k, v in list(event.features.items())[:12]:
            if isinstance(v, float):
                print(f'  {k:<28}: {v:.4f}')
            else:
                print(f'  {k:<28}: {v}')


def make_static_vector(base='median', raw_overrides=None):
    """Create a schema-consistent Layer 1 vector aligned to X.columns."""
    raw_overrides = raw_overrides or {}
    if base == 'low':
        raw = X[RAW_FEATURE_COLUMNS].quantile(0.25).to_frame().T
    elif base == 'high':
        raw = X[RAW_FEATURE_COLUMNS].quantile(0.75).to_frame().T
    else:
        raw = X[RAW_FEATURE_COLUMNS].median().to_frame().T

    for col, val in raw_overrides.items():
        if col in raw.columns:
            raw.loc[raw.index[0], col] = float(val)
        else:
            print(f'  warning: raw feature {col} not found; ignored')

    engineered = engineer_features(raw)
    # Align to the exact training/inference schema.
    for col in X.columns:
        if col not in engineered.columns:
            engineered[col] = 0.0
    return engineered[X.columns].iloc[0].astype(float).values


def benign_notepad_stream(n=48, seed=42):
    rng = np.random.default_rng(seed)
    rows = []
    for _ in range(n):
        rows.append({
            'cpu_percent'          : float(rng.uniform(0.5, 5.0)),
            'memory_rss_mb'        : float(rng.uniform(40, 90)),
            'memory_vms_mb'        : float(rng.uniform(160, 260)),
            'io_read_bytes_delta'  : float(rng.uniform(0, 2_000)),
            'io_write_bytes_delta' : float(rng.uniform(0, 2_000)),
            'net_bytes_sent_delta' : float(rng.uniform(0, 100)),
            'num_open_files'       : float(rng.integers(0, 4)),
        })
    return rows


def ransomware_behavior_stream(n=56, encrypt_start=12, seed=7):
    rng = np.random.default_rng(seed)
    rows = []
    for i in range(n):
        encrypting = i >= encrypt_start
        rows.append({
            'cpu_percent'          : float(rng.uniform(82, 99) if encrypting else rng.uniform(1, 5)),
            'memory_rss_mb'        : float(rng.uniform(120, 260)),
            'memory_vms_mb'        : float(rng.uniform(350, 750)),
            'io_read_bytes_delta'  : float(rng.uniform(2_000_000, 8_000_000) if encrypting else rng.uniform(0, 5_000)),
            'io_write_bytes_delta' : float(rng.uniform(4_000_000, 15_000_000) if encrypting else rng.uniform(0, 4_000)),
            'net_bytes_sent_delta' : float(rng.uniform(0, 500)),
            'num_open_files'       : float(rng.integers(80, 200) if encrypting else rng.integers(1, 5)),
        })
    return rows


In [ ]:
# ── TEST 1: benign Notepad-like activity ────────────────────────────────────
if EDR_OK:
    benign_static = make_static_vector(base='low')
    result_notepad = orchestrator.evaluate(
        static_features  = benign_static,
        telemetry_stream = benign_notepad_stream(),
        pid              = 4444,
        process_name     = 'notepad.exe',
        feature_cols      = CNN_FEATURE_COLS,
    )
    print_event(result_notepad, 'TEST 1 — Benign Notepad-like process')

    hard_actions = {'kill_process', 'block_network', 'snapshot_directory'}
    if hard_actions.intersection(set(result_notepad.recommended_actions)):
        print('\n✗ REGRESSION: benign process received hard mitigation.')
    else:
        print('\n✓ PASS: benign process was not hard-blocked.')
else:
    print('EDR not ready — run previous cells first.')


In [ ]:
# ── TEST 2: behavioral ransomware with clean/neutral memory ─────────────────
if EDR_OK:
    clean_memory_static = make_static_vector(base='median')
    result_behavior = orchestrator.evaluate(
        static_features  = clean_memory_static,
        telemetry_stream = ransomware_behavior_stream(),
        pid              = 5555,
        process_name     = 'unknown_encryptor.exe',
        feature_cols      = CNN_FEATURE_COLS,
    )
    print_event(result_behavior, 'TEST 2 — Dynamic encryption behavior')

    if result_behavior.alert:
        print('\n✓ PASS: dynamic behavior produced an EDR alert/mitigation path.')
    else:
        print('\n⚠ No alert. Inspect CNN threshold, evidence thresholds, and strike settings.')
else:
    print('EDR not ready — run previous cells first.')


In [ ]:
# ── TEST 3: memory-forensics ransomware signal with engineered features recomputed ──
if EDR_OK:
    memory_overrides = {
        'malfind.ninjections'     : 15.0,
        'malfind.commitCharge'    : 90.0,
        'malfind.uniqueInjections': 10.0,
        'ldrmodules.not_in_load'  : 30.0,
        'ldrmodules.not_in_mem'   : 20.0,
        'psxview.not_in_pslist'   : 5.0,
        'handles.nfile'           : 2500.0,
        'handles.nhandles'        : 12000.0,
    }
    suspicious_static = make_static_vector(base='median', raw_overrides=memory_overrides)
    quiet_stream = benign_notepad_stream(n=24, seed=123)

    result_memory = orchestrator.evaluate(
        static_features  = suspicious_static,
        telemetry_stream = quiet_stream,
        pid              = 6666,
        process_name     = 'memory_suspicious.exe',
        feature_cols      = CNN_FEATURE_COLS,
    )
    print_event(result_memory, 'TEST 3 — Static memory-forensics signal')

    print('\nNote: this synthetic test recomputes engineered features from raw overrides.')
    print('It is a better Layer 1 test than manually assigning inconsistent engineered values.')
else:
    print('EDR not ready — run previous cells first.')


### 17.7 — Process-Level Metrics and Detection Latency

Window-level metrics are useful, but EDR decisions happen at the process level. The improved notebook therefore reports both CNN window-level metrics and orchestrator process-level behavior in the synthetic regression tests above.


In [ ]:
# ── 17.7 Window-level CNN summary and process-level reminder ────────────────
print('='*72)
print('IMPROVED LAYER 2 / EDR SUMMARY — V7')
print('='*72)

if CNN_TRAINED:
    print('Window-level CNN metrics:')
    for k in ['recall', 'precision', 'f1', 'roc_auc', 'pr_auc', 'fpr', 'specificity', 'balanced_accuracy', 'threshold']:
        print(f'  {k:<18}: {cnn_results.get(k)}')

    print('\nImportant interpretation:')
    print('  CNN probability is only a behavioral suspicion score.')
    print('  EDR mitigation requires state + persistence/risk accumulation + encryption evidence.')
    print('  A high window-level FPR means hard blocking must stay behind the orchestrator gate.')
else:
    print('CNN was not trained in this run.')

if EDR_OK:
    print('\nProcess-level regression test outcomes available as:')
    for name in ['result_notepad', 'result_behavior', 'result_memory']:
        if name in globals():
            ev = globals()[name]
            print(f'  {name:<16}: alert={ev.alert} severity={ev.severity} source={ev.model_source}')



## 18 — Explainability: SHAP for Layer 1 and Occlusion for Layer 2

This section adds explainability support:

- **Layer 1:** `static_shap_explanation()` uses TreeSHAP if available and falls back to feature importances.
- **Layer 2:** `cnn_occlusion_explanation()` hides one telemetry feature at a time and measures how much the CNN probability drops.

The goal is to explain alerts in terms of security evidence, not only probability numbers.


In [ ]:
# ── 18.1 Layer 2 CNN occlusion explanation ─────────────────────────────────
if CNN_TRAINED:
    from src.explainability.edr_explain import cnn_occlusion_explanation

    # Use a moderate synthetic ransomware-behavior window. Fully saturated P≈1
    # windows are poor explanation examples because occlusion may not move them.
    raw_stream = ransomware_behavior_stream(n=20, encrypt_start=4, seed=99)
    raw_window = []
    for tick in raw_stream[4:4 + CNN_WINDOW_SIZE]:
        tick2 = orchestrator._ensure_tick_features(tick, CNN_FEATURE_COLS) if EDR_OK else tick
        raw_window.append([float(tick2[col]) for col in CNN_FEATURE_COLS])
    raw_window = np.array(raw_window, dtype=np.float32)

    occ = cnn_occlusion_explanation(
        cnn_model    = cnn,
        window_raw   = raw_window,
        feature_cols = CNN_FEATURE_COLS,
        scaler       = cnn.scaler,
        baseline     = 'scaler_center',
        top_k        = 10,
        coherent     = True,
    )
    occ_df = pd.DataFrame(occ)
    print('Top CNN coherent occlusion explanations:')
    print('  Note: prefer non-saturated probabilities for final report examples.')
    display(occ_df)
else:
    print('CNN not trained — occlusion skipped.')


In [ ]:
# ── 18.2 Layer 1 static explanation ─────────────────────────────────────────
if EDR_OK:
    from src.explainability.edr_explain import static_shap_explanation

    static_model_for_explain = orchestrator.static_model
    feature_dict = dict(zip(static_model_for_explain.feature_names, suspicious_static)) if 'suspicious_static' in globals() else dict(zip(static_model_for_explain.feature_names, make_static_vector()))

    static_exp = static_shap_explanation(static_model_for_explain, feature_dict, top_k=12)
    if static_exp:
        print('Top Layer 1 memory explanation features:')
        display(pd.DataFrame(static_exp))
    else:
        print('No SHAP/feature-importance explanation available for this static model object.')
else:
    print('EDR not ready — Layer 1 explanation skipped.')


## 19 — Counterfactual Validation

Counterfactual tests check whether the CNN reacts to encryption-like behavior rather than lab artifacts.

Expected behavior:

- for a ransomware-like window, suppressing read/write/CPU-write features should reduce probability;
- for a benign window, injecting encryption-like features should increase probability.


In [ ]:
# ── 19.1 Counterfactual tests for encryption behavior ───────────────────────
if CNN_TRAINED:
    from src.evaluation.counterfactual_tests import run_counterfactual_pair

    # Ransomware-like window: active encryption starts inside the sampled window.
    ran_stream = ransomware_behavior_stream(n=18, encrypt_start=4, seed=202)
    ran_window = []
    for tick in ran_stream[4:4 + CNN_WINDOW_SIZE]:
        tick2 = orchestrator._ensure_tick_features(tick, CNN_FEATURE_COLS) if EDR_OK else tick
        ran_window.append([float(tick2[col]) for col in CNN_FEATURE_COLS])
    ran_window = np.array(ran_window, dtype=np.float32)

    # Benign-like window
    ben_stream = benign_notepad_stream(n=16, seed=303)
    ben_window = []
    for tick in ben_stream[:CNN_WINDOW_SIZE]:
        tick2 = orchestrator._ensure_tick_features(tick, CNN_FEATURE_COLS) if EDR_OK else tick
        ben_window.append([float(tick2[col]) for col in CNN_FEATURE_COLS])
    ben_window = np.array(ben_window, dtype=np.float32)

    cf_ran = run_counterfactual_pair(cnn, ran_window, CNN_FEATURE_COLS, scaler=cnn.scaler)
    cf_ben = run_counterfactual_pair(cnn, ben_window, CNN_FEATURE_COLS, scaler=cnn.scaler)

    cf_df = pd.DataFrame([
        {'window': 'ransomware_like', **cf_ran},
        {'window': 'benign_like', **cf_ben},
    ])
    display(cf_df)

    print('Interpretation:')
    print('  ransomware_like suppression_drop should be positive.')
    print('  benign_like injection_increase should be positive.')
    print('  If either direction fails, do not use the counterfactual as proof; treat it as a diagnostic.')
else:
    print('CNN not trained — counterfactual tests skipped.')


## 20 — Updated Final Notes

After running this updated notebook, compare the new results to the old run:

- The old result had high recall but a very high false-positive rate.
- The updated pipeline should first verify label correctness, then retrain Layer 2.
- If the CNN still has a high window-level FPR, the orchestrator must continue treating it as suspicion only.
- Hard-benign negative controls are still required before claiming production readiness.

Recommended next dataset additions:

- 7-Zip compression of many files;
- backup or sync tools;
- large folder copy;
- antivirus scan;
- installer/updater;
- compiler build;
- browser large download;
- Office autosave / Notepad save loops.
